## **SCRIPT 1:**  - Feature Engineering Engine


*   Run daily before running script 3




In [ ]:
# ==============================================================================
# SCRIPT 1: Feature Engineering Engine (v6.2 - FINAL WATER-TIGHT VERSION)
# ==============================================================================
!pip install -q pandas unidecode tqdm scikit-learn

import pandas as pd
import numpy as np
import os
import re
import pytz
from datetime import datetime
from google.colab import drive
from unidecode import unidecode
from tqdm import tqdm

def load_historical_odds(file_path):
    print("--- Loading and Processing Historical Odds Data ---")
    try: df = pd.read_csv(file_path)
    except FileNotFoundError: print(f"CRITICAL ERROR: File not found at {file_path}"); return pd.DataFrame()
    team_map = {'atl': 'ATL', 'bkn': 'BKN', 'bos': 'BOS', 'cha': 'CHA', 'chi': 'CHI', 'cle': 'CLE', 'dal': 'DAL', 'den': 'DEN', 'det': 'DET', 'gs': 'GSW', 'hou': 'HOU', 'ind': 'IND', 'lac': 'LAC', 'lal': 'LAL', 'mem': 'MEM', 'mia': 'MIA', 'mil': 'MIL', 'min': 'MIN', 'no': 'NOP', 'ny': 'NYK', 'okc': 'OKC', 'orl': 'ORL', 'phi': 'PHI', 'phx': 'PHX', 'por': 'POR', 'sa': 'SAS', 'sac': 'SAC', 'tor': 'TOR', 'utah': 'UTA', 'wsh': 'WAS'}
    df['home'] = df['home'].str.lower().map(team_map); df['away'] = df['away'].str.lower().map(team_map)
    df.dropna(subset=['home', 'away'], inplace=True)
    df['GAME_DATE'] = pd.to_datetime(df['date']); df['VEGAS_SPREAD'] = np.where(df['whos_favored'] == 'home', -df['spread'], df['spread']); df.rename(columns={'total': 'VEGAS_TOTAL'}, inplace=True)
    df['game_pair'] = [frozenset(x) for x in zip(df['home'], df['away'])]; print("✅ Historical odds data processed successfully.")
    return df[['GAME_DATE', 'game_pair', 'VEGAS_SPREAD', 'VEGAS_TOTAL']]

def get_current_nba_season():
    # Use actual current date instead of hardcoded 2025
    today = datetime.now()
    year, month = today.year, today.month
    # Returns '2025-26' if we are in May 2026
    return f"{year}-{str(year+1)[-2:]}" if month >= 9 else f"{year-1}-{str(year)[-2:]}"

def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except (ValueError, TypeError): return np.nan
    return pd.to_numeric(min_str, errors='coerce')

def calculate_weighted_rolling_averages(group, stats_to_roll):
    avg_minutes = group['MIN'].mean()
    weights = np.ones(len(group)); weights[group['MIN'] < 0.7 * avg_minutes] = 0.5; weights[group['MIN'] > 1.15 * avg_minutes] = 1.2; group['WEIGHT'] = weights
    for stat in stats_to_roll:
        # Internally shifts by 1 to ensure rolling average for 'today' only uses 'yesterday' and back
        weighted_stat = group[stat] * group['WEIGHT']
        for w in [5, 10, 20]:
            rolling_weighted_sum = weighted_stat.shift(1).rolling(w, min_periods=1).sum()
            rolling_weight_sum = group['WEIGHT'].shift(1).rolling(w, min_periods=1).sum()
            group[f'W_ROLL_AVG_{stat}_{w}'] = rolling_weighted_sum / rolling_weight_sum
    return group

def process_nba_data_v6():
    drive.mount('/content/drive', force_remount=True); BASE_DIR = "/content/drive/MyDrive/NBA/"; DATA_DIR = os.path.join(BASE_DIR, "Historical Data/"); BOXSCORE_DIR = os.path.join(DATA_DIR, "NocturneBear_Boxscores")
    odds_df = load_historical_odds(os.path.join(DATA_DIR, "nba_2008-2025.csv"));
    if odds_df.empty: return
    print("\nStep 2: Loading and cleaning historical box score data...")
    df = pd.concat([pd.read_csv(os.path.join(BOXSCORE_DIR, f), low_memory=False) for f in os.listdir(BOXSCORE_DIR) if f.endswith('.csv')], ignore_index=True)
    rename_map = {'season_year': 'SEASON_YEAR', 'game_date': 'GAME_DATE', 'gameId': 'GAME_ID', 'personId': 'PLAYER_ID', 'personName': 'PLAYER_NAME', 'teamTricode': 'TEAM_ABBREVIATION', 'points': 'PTS', 'reboundsTotal': 'REB', 'assists': 'AST', 'minutes': 'MIN', 'fieldGoalsAttempted': 'FGA', 'fieldGoalsMade': 'FGM', 'freeThrowsAttempted': 'FTA', 'threePointersAttempted': 'FG3A', 'threePointersMade': 'FG3M', 'steals': 'STL', 'blocks': 'BLK', 'turnovers': 'TOV'}
    df.rename(columns=rename_map, inplace=True); CURRENT_SEASON_STR = get_current_nba_season()

    # FINAL FIX: Changed < to <= to include the current 2025-26 season data
    df = df[(df['SEASON_YEAR'] >= '2015-16') & (df['SEASON_YEAR'] <= CURRENT_SEASON_STR)].copy()

    df = df[df['MIN'].notna() & (df['MIN'] != '0:00') & (df['MIN'] != '0')].copy(); df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']); df['MIN'] = df['MIN'].apply(convert_min_to_decimal)
    numeric_cols = ['PTS', 'REB', 'AST', 'MIN', 'FGM', 'FGA', 'FTA', 'FG3A', 'FG3M', 'STL', 'BLK', 'TOV'];
    for col in numeric_cols: df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=numeric_cols, inplace=True)

    # Sort for shifting
    df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)

    # Raw Efficiency Metrics
    df['EFG_PCT_RAW'] = (df['FGM'] + 0.5 * df['FG3M']) / df['FGA'].replace(0, np.nan)
    df['TS_PCT_RAW'] = df['PTS'] / (2 * (df['FGA'] + 0.44 * df['FTA'])).replace(0, np.nan)
    df['EFG_PCT_RAW'] = df['EFG_PCT_RAW'].fillna(0); df['TS_PCT_RAW'] = df['TS_PCT_RAW'].fillna(0)

    print("\nStep 3: Merging historical odds data...")
    opponent_df = df[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates(); df = pd.merge(df, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID'); df = df[df['TEAM_ABBREVIATION'] != df['OPPONENT_ABBR']].copy()
    df['game_pair'] = [frozenset(x) for x in zip(df['TEAM_ABBREVIATION'], df['OPPONENT_ABBR'])]; df = pd.merge(df, odds_df, on=['GAME_DATE', 'game_pair'], how='left'); df.drop(columns=['game_pair'], inplace=True)

    print("\nStep 4: Engineering Functional Role Ratios (SHIFTED)...")
    stats_for_ratios = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FTA', 'TOV', 'FG3M']
    for stat in stats_for_ratios:
        # Calculate ratio, then shift by 1 within player group to prevent leakage
        df[f'{stat}_PM'] = (df[stat] / df['MIN'].replace(0, np.nan))
        df[f'{stat}_PM'] = df.groupby('PLAYER_ID')[f'{stat}_PM'].shift(1)

    # Prev Efficiency features
    df['EFG_PCT_PREV'] = df.groupby('PLAYER_ID')['EFG_PCT_RAW'].shift(1)
    df['TS_PCT_PREV'] = df.groupby('PLAYER_ID')['TS_PCT_RAW'].shift(1)

    print("\nStep 5: Engineering Weighted Rolling Averages & Trends...")
    # Passing RAW metrics into rolling avg; function will handle the shift internally
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FG3M', 'FG3A', 'EFG_PCT_RAW', 'TS_PCT_RAW']
    df = df.groupby('PLAYER_ID', group_keys=False).progress_apply(calculate_weighted_rolling_averages, stats_to_roll=stats_to_roll)
    for stat in stats_to_roll: df[f'TREND_{stat}'] = df[f'W_ROLL_AVG_{stat}_5'] - df[f'W_ROLL_AVG_{stat}_20']
    df['DAYS_OF_REST'] = df.groupby('PLAYER_ID')['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1).clip(upper=7).fillna(7)

    print("\nStep 6: Engineering Opponent Context Features...")
    team_allowed_stats = df.groupby(['GAME_ID', 'OPPONENT_ABBR']).agg(GAME_DATE=('GAME_DATE', 'first'), PTS=('PTS', 'sum'), REB=('REB', 'sum'), AST=('AST', 'sum'), FG3M=('FG3M', 'sum'), EFG_PCT=('EFG_PCT_RAW', 'mean')).reset_index(); team_allowed_stats.rename(columns={'OPPONENT_ABBR': 'TEAM'}, inplace=True); team_allowed_stats.sort_values(by=['TEAM', 'GAME_DATE'], inplace=True)
    for stat in ['PTS', 'REB', 'AST', 'FG3M', 'EFG_PCT']:
        team_allowed_stats[f'OPP_ALLOWED_{stat}_AVG_30'] = team_allowed_stats.groupby('TEAM')[stat].transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    df = pd.merge(df, team_allowed_stats[['GAME_ID', 'TEAM', f'OPP_ALLOWED_PTS_AVG_30', f'OPP_ALLOWED_REB_AVG_30', f'OPP_ALLOWED_AST_AVG_30', f'OPP_ALLOWED_FG3M_AVG_30', f'OPP_ALLOWED_EFG_PCT_AVG_30']], left_on=['GAME_ID', 'OPPONENT_ABBR'], right_on=['GAME_ID', 'TEAM'], how='left')

    print("\nStep 7: Finalizing data and creating target variables...")
    for stat in ['MIN', 'PTS', 'REB', 'AST', 'FG3M']: df[f'TARGET_{stat}'] = df[stat]
    dd_td_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']; df['DD_COUNT'] = df[dd_td_stats].ge(10).sum(axis=1); df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int); df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)

    # We only use features that end in _PM, _PREV, _ROLL_AVG, or TREND (all of which are now shifted)
    feature_columns = [c for c in df.columns if 'W_ROLL_AVG' in c or 'TREND' in c or 'OPP_ALLOWED' in c or '_PM' in c or 'VEGAS' in c or '_PREV' in c] + ['DAYS_OF_REST']; target_columns = [c for c in df.columns if 'TARGET' in c]
    final_df = df[['GAME_DATE', 'PLAYER_ID', 'PLAYER_NAME'] + feature_columns + target_columns].copy(); final_df.dropna(subset=[c for c in feature_columns if 'W_ROLL_AVG' in c], inplace=True)

    output_path = os.path.join(DATA_DIR, "nba_advanced_training_data_v6.csv"); final_df.to_csv(output_path, index=False); print(f"\n✅ All Leaks Sealed. Model-ready rows: {len(final_df)} saved to: {output_path}"); display(final_df.head())

# Get current time in EST
    est = pytz.timezone('US/Eastern')
    now = datetime.now(est)
    formatted_time = now.strftime("%-I:%M %p").lower().replace("am", "a.m.").replace("pm", "p.m.")
    formatted_date = now.strftime("%-m-%-d-%y")
    print(f"Completed at: {formatted_time} {formatted_date} EST")

    display(final_df.head())

if __name__ == "__main__":
    tqdm.pandas(); process_nba_data_v6()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.5 MB/s eta 0:00:00


ValueError: mount failed

**SCRIPT 2:** - Model Training

In [ ]:
# ==============================================================================
# SCRIPT 2: Model Training Engine (v7.2 - LEAK-FREE & CLEAN OUTPUT)
# ==============================================================================
!pip install -q lightgbm scikit-learn pandas joblib optuna tqdm

import pandas as pd
import numpy as np
import joblib
import os
import json
import lightgbm as lgb
import optuna
import warnings
from tqdm import tqdm
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold

# Suppress feature name warnings and Optuna verbosity
warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Optuna Objective Function ---
def objective_lgbm(trial, X, y):
    params = {
        'objective': 'rmse',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)
    }

    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []

    # Fast Cross-Validation for Tuning
    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_fold, y_train_fold)
        preds = model.predict(X_val_fold)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))

    return np.mean(rmse_scores)

def run_optuna_tuning(X, y, target_name, n_trials=20):
    print(f"   >>> Tuning Hyperparameters for {target_name}...")
    study = optuna.create_study(direction='minimize')

    # Progress bar for trials
    with tqdm(total=n_trials, desc=f"      {target_name} Tuning", unit="trial", leave=True) as pbar:
        def callback(study, trial):
            pbar.update(1)
        study.optimize(lambda trial: objective_lgbm(trial, X, y), n_trials=n_trials, callbacks=[callback])

    print(f"   ✅ Best params for {target_name}: {study.best_params}")
    return study.best_params

def train_and_validate_models_v7():
    drive.mount('/content/drive', force_remount=True); BASE_DIR = "/content/drive/MyDrive/NBA/"; DATA_DIR = os.path.join(BASE_DIR, "Historical Data/"); MODEL_DIR = os.path.join(BASE_DIR, "Training Model Advanced"); os.makedirs(MODEL_DIR, exist_ok=True)
    df = pd.read_csv(os.path.join(DATA_DIR, "nba_advanced_training_data_v6.csv")); df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']); print(f"Loaded {len(df)} rows from v6 file.")

    print("\n--- Performing Chronological Train/Test Split ---")
    df = df.sort_values('GAME_DATE'); split_date = df['GAME_DATE'].max() - pd.DateOffset(months=2)
    # Added .copy() to ensure these are independent objects and prevent warnings
    train_df = df[df['GAME_DATE'] < split_date].copy(); test_df = df[df['GAME_DATE'] >= split_date].copy()
    print(f"Training data: {len(train_df)} rows | Test data: {len(test_df)} rows")
    mae_values = {}

    print("\n--- STAGE 1: Training & Validating Minutes Model (w/ Optuna) ---")
    min_features = [c for c in df.columns if 'W_ROLL_AVG_MIN' in c or 'TREND_MIN' in c] + ['DAYS_OF_REST']
    X_train_min, y_train_min = train_df[min_features], train_df['TARGET_MIN']; X_test_min, y_test_min = test_df[min_features], test_df['TARGET_MIN']

    imputer_min = SimpleImputer(strategy='mean'); X_train_min_imputed = imputer_min.fit_transform(X_train_min)

    # *** OPTUNA TUNING FOR MINUTES ***
    best_params_min = run_optuna_tuning(X_train_min_imputed, y_train_min, "Minutes")
    minutes_model = lgb.LGBMRegressor(**best_params_min, random_state=42)
    minutes_model.fit(X_train_min_imputed, y_train_min)

    X_test_min_imputed = imputer_min.transform(X_test_min); preds_min = minutes_model.predict(X_test_min_imputed)
    mae_values['MIN'] = mean_absolute_error(y_test_min, preds_min); print(f"✅ Minutes Model Validation RMSE: {np.sqrt(mean_squared_error(y_test_min, preds_min)):.3f} | MAE: {mae_values['MIN']:.3f}")

    print("\n--- STAGE 2: Training & Validating Core Stats Models (w/ Optuna) ---")
    # Adding predicted minutes as a feature for stats
    train_df['Proj_MIN'] = minutes_model.predict(X_train_min_imputed); test_df['Proj_MIN'] = minutes_model.predict(X_test_min_imputed)

    # Feature selection logic to include new Shot Quality (_PREV) features from Script 1
    stat_features = [c for c in df.columns if ('W_ROLL_AVG' in c or 'TREND' in c or 'OPP_ALLOWED' in c or '_PM' in c or 'VEGAS' in c or '_PREV' in c) and 'MIN' not in c] + ['DAYS_OF_REST', 'Proj_MIN']

    X_train_stats = train_df[stat_features]; X_test_stats = test_df[stat_features]
    imputer_stats = SimpleImputer(strategy='mean'); X_train_stats_imputed = imputer_stats.fit_transform(X_train_stats); X_test_stats_imputed = imputer_stats.transform(X_test_stats)

    test_preds = {}
    best_params_stats = {}

    # *** OPTUNA TUNING FOR STATS ***
    for prop in ['PTS', 'REB', 'AST', 'FG3M']:
        best_params_stats[prop] = run_optuna_tuning(X_train_stats_imputed, train_df[f'TARGET_{prop}'], prop)
        model = lgb.LGBMRegressor(**best_params_stats[prop], random_state=42)
        model.fit(X_train_stats_imputed, train_df[f'TARGET_{prop}'])

        preds_stat = model.predict(X_test_stats_imputed); test_preds[prop] = preds_stat
        mae_values[prop] = mean_absolute_error(test_df[f'TARGET_{prop}'], preds_stat)
        print(f"  - {prop} Model Validation RMSE: {np.sqrt(mean_squared_error(test_df[f'TARGET_{prop}'], preds_stat)):.3f} | MAE: {mae_values[prop]:.3f}")

    # Calculate combination MAEs (PRA, etc.)
    mae_values['PRA'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_REB'] + test_df['TARGET_AST'], test_preds['PTS'] + test_preds['REB'] + test_preds['AST'])
    mae_values['P+R'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_REB'], test_preds['PTS'] + test_preds['REB'])
    mae_values['P+A'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_AST'], test_preds['PTS'] + test_preds['AST'])
    mae_values['R+A'] = mean_absolute_error(test_df['TARGET_REB'] + test_df['TARGET_AST'], test_preds['REB'] + test_preds['AST'])

    with open(os.path.join(MODEL_DIR, "mae_values.json"), 'w') as f: json.dump(mae_values, f, indent=4);
    print(f"✅ MAE values saved to mae_values.json")

    print("\n--- Finalizing Production Models (Re-training on all data) ---")
    imputer_min_final = SimpleImputer(strategy='mean'); X_min_full_imputed = imputer_min_final.fit_transform(df[min_features])

    minutes_model_final = lgb.LGBMRegressor(**best_params_min, random_state=42)
    minutes_model_final.fit(X_min_full_imputed, df['TARGET_MIN'])

    joblib.dump(minutes_model_final, os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")); joblib.dump(imputer_min_final, os.path.join(MODEL_DIR, "minutes_imputer.pkl")); joblib.dump(min_features, os.path.join(MODEL_DIR, "minutes_feature_names.pkl"))

    df['Proj_MIN'] = minutes_model_final.predict(X_min_full_imputed)
    imputer_stats_final = SimpleImputer(strategy='mean'); X_stats_full_imputed = imputer_stats_final.fit_transform(df[stat_features])
    joblib.dump(imputer_stats_final, os.path.join(MODEL_DIR, "stats_imputer.pkl")); joblib.dump(stat_features, os.path.join(MODEL_DIR, "stats_feature_names.pkl"))

    for prop in ['PTS', 'REB', 'AST', 'FG3M']:
        model_final = lgb.LGBMRegressor(**best_params_stats[prop], random_state=42)
        model_final.fit(X_stats_full_imputed, df[f'TARGET_{prop}'])
        joblib.dump(model_final, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_model.pkl"))
    print("✅ Final Stats models saved.")

    df['Proj_PTS'] = joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")).predict(X_stats_full_imputed)
    df['Proj_REB'] = joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")).predict(X_stats_full_imputed)
    df['Proj_AST'] = joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")).predict(X_stats_full_imputed)
    classification_features = ['Proj_PTS', 'Proj_REB', 'Proj_AST']

    for prop in ['DD', 'TD']:
        y_class_full = df[f'TARGET_{prop}']
        if y_class_full.sum() > 50:
            classifier_final = lgb.LGBMClassifier(objective='binary', class_weight='balanced', random_state=42); classifier_final.fit(df[classification_features], y_class_full)
            joblib.dump(classifier_final, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_model.pkl"))
    print("✅ Final Classification models saved.")

if __name__ == "__main__":
    train_and_validate_models_v7()

Mounted at /content/drive
Loaded 243008 rows from v6 file.

--- Performing Chronological Train/Test Split ---
Training data: 241111 rows | Test data: 1897 rows

--- STAGE 1: Training & Validating Minutes Model (w/ Optuna) ---
   >>> Tuning Hyperparameters for Minutes...


      Minutes Tuning: 100%|██████████| 20/20 [06:57<00:00, 20.85s/trial]


   ✅ Best params for Minutes: {'n_estimators': 117, 'learning_rate': 0.05211132825754309, 'num_leaves': 84, 'max_depth': 5, 'min_child_samples': 40, 'subsample': 0.9064540837122829, 'colsample_bytree': 0.8269350874523901, 'reg_alpha': 0.30404143569666064, 'reg_lambda': 0.2712598622292623}
✅ Minutes Model Validation RMSE: 6.427 | MAE: 5.240

--- STAGE 2: Training & Validating Core Stats Models (w/ Optuna) ---
   >>> Tuning Hyperparameters for PTS...


      PTS Tuning: 100%|██████████| 20/20 [21:51<00:00, 65.57s/trial]


   ✅ Best params for PTS: {'n_estimators': 171, 'learning_rate': 0.06439299394843724, 'num_leaves': 42, 'max_depth': 5, 'min_child_samples': 61, 'subsample': 0.8010443736809774, 'colsample_bytree': 0.7410052406141762, 'reg_alpha': 0.9761937703997426, 'reg_lambda': 0.5868857394588054}
  - PTS Model Validation RMSE: 5.722 | MAE: 4.423
   >>> Tuning Hyperparameters for REB...


...

   ✅ Best params for REB: {'n_estimators': 411, 'learning_rate': 0.07138628340512618, 'num_leaves': 48, 'max_depth': 3, 'min_child_samples': 100, 'subsample': 0.6696596280518274, 'colsample_bytree': 0.674460089195546, 'reg_alpha': 0.3048622230106589, 'reg_lambda': 0.6029812302412497}
  - REB Model Validation RMSE: 2.454 | MAE: 1.853
   >>> Tuning Hyperparameters for AST...


      AST Tuning: 100%|██████████| 20/20 [17:07<00:00, 51.36s/trial]


   ✅ Best params for AST: {'n_estimators': 233, 'learning_rate': 0.057619309204685804, 'num_leaves': 23, 'max_depth': 7, 'min_child_samples': 20, 'subsample': 0.8365738934647806, 'colsample_bytree': 0.5110841696196774, 'reg_alpha': 0.8758075981478164, 'reg_lambda': 0.7225335173184223}
  - AST Model Validation RMSE: 1.631 | MAE: 1.238
   >>> Tuning Hyperparameters for FG3M...


      FG3M Tuning: 100%|██████████| 20/20 [20:22<00:00, 61.11s/trial]


   ✅ Best params for FG3M: {'n_estimators': 885, 'learning_rate': 0.01168210791782098, 'num_leaves': 23, 'max_depth': 9, 'min_child_samples': 45, 'subsample': 0.7327891690713453, 'colsample_bytree': 0.6760060295122322, 'reg_alpha': 0.559735945226685, 'reg_lambda': 0.7958090532360078}
  - FG3M Model Validation RMSE: 1.227 | MAE: 0.890
✅ MAE values saved to mae_values.json

--- Finalizing Production Models (Re-training on all data) ---
✅ Final Stats models saved.
✅ Final Classification models saved.


## **Script 3** - NBA Professional Sharp & ML Engine

In [ ]:
# ==============================================================================
# SCRIPT 3: NBA Professional Sharp & ML Engine (v125 - THE UNABRIDGED MASTER - RESTORED)
# Features: 10-Game Streaks, Robust H2H, CLV/RLM/Steam on ALL Sheets, Full Guides
# NEW: H2H Crusher Logic (Dort Rule), Tightened My Way Alt-Lines (0.75 MAE)
# NEW: Star Power Resilience, MSG Factor (Venue Boost), Targeted Defense
# FIX: H2H Volume Fix (Strict Token Match), Blank Cell Population Fix
# FIX: Odds API Timestamp Format Fix, TypeError & No Props Fix, ScoreboardV3
# ==============================================================================

# --- Part 0: Setup & Installs ---
!pip install -q xlsxwriter nba_api scikit-learn pandas joblib pytz unidecode requests lightgbm python-dateutil tqdm openpyxl scipy

import os
import re
import time
import joblib
import warnings
import requests
import json
import traceback
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from dateutil import parser
from unidecode import unidecode
from tqdm import tqdm
from decimal import Decimal
from scipy.stats import poisson, norm
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display, clear_output
from xlsxwriter.utility import xl_col_to_name

# UPDATED IMPORT: Preserving all necessary endpoints + ScoreboardV3 + NBAHTTP
from nba_api.stats.endpoints._base import Endpoint
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.library.http import NBAHTTP
from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats, teamestimatedmetrics, drafthistory, ScoreboardV3, scoreboardv2, leaguedashteamstats, leaguegamefinder, leaguedashplayerptshot, synergyplaytypes, leaguedashptteamdefend
from nba_api.stats.static import teams, players

warnings.filterwarnings("ignore")

# 1. Inject required browser headers ONLY for NBAHTTP (cdn.nba.com) to fix ScoreboardV3.
# We DO NOT touch NBAStatsHTTP so Defense Ranks & Synergy work flawlessly.
NBAHTTP.headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept-Language': 'en-US,en;q=0.9',
}

# --- Part 1: Configuration ---
GDRIVE_BASE_PATH = "/content/drive/MyDrive/NBA/"
MODEL_DIR = os.path.join(GDRIVE_BASE_PATH, "Training Model Advanced/")
OUTPUT_DIR = os.path.join(GDRIVE_BASE_PATH, "Predictions/")
GAMELOG_LAKE_DIR = os.path.join(GDRIVE_BASE_PATH, "gamelog_lake/")
DATA_DIR = os.path.join(GDRIVE_BASE_PATH, "Historical Data/")
INITIAL_LINES_PATH = os.path.join(OUTPUT_DIR, "initial_nba_lines.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(GAMELOG_LAKE_DIR, exist_ok=True)

# *** 1. MARKET TOGGLES ***
ENABLE_MARKETS = {
    "Moneyline": True, "Spreads": True, "Totals": True,
    "Points": True, "Rebounds": True, "Assists": True, "Threes": True,
    "Combos": True, "DoubleDouble": True, "TripleDouble": True
}

FILTER_OUT_PLAYERS = True

_GAME_MAP = {"Moneyline": "h2h", "Spreads": "spreads", "Totals": "totals"}
_PROP_MAP = {
    "Points": {'player_points': 'PTS'}, "Rebounds": {'player_rebounds': 'REB'}, "Assists": {'player_assists': 'AST'},
    "Threes": {'player_threes': 'FG3M'},
    "Combos": {'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A'},
    "DoubleDouble": {'player_double_double': 'DD'}, "TripleDouble": {'player_triple_double': 'TD'}
}

# [FIX] Simple Map for Column Lookups to prevent "Way Off" Projections
SIMPLE_PROP_MAP = {
    "Points": "PTS", "PTS": "PTS",
    "Rebounds": "REB", "REB": "REB",
    "Assists": "AST", "AST": "AST",
    "Threes": "FG3M", "FG3M": "FG3M", "3PT": "FG3M",
    "Combos": "PRA", "PRA": "PRA",
    "DoubleDouble": "DD", "DD": "DD",
    "TripleDouble": "TD", "TD": "TD",
    "P+R": "P+R", "P+A": "P+A", "R+A": "R+A"
}

ACTIVE_GAME_MARKETS =[v for k, v in _GAME_MAP.items() if ENABLE_MARKETS.get(k, False)]

# STRICT MAPPING: Prevents cross-contamination of single stats and combos
MARKET_NAME_MAP = {}
for category, enabled in ENABLE_MARKETS.items():
    if enabled and category in _PROP_MAP:
        MARKET_NAME_MAP.update(_PROP_MAP[category])
ALL_PROP_MARKETS_LIST = list(MARKET_NAME_MAP.keys())

# *** EXACT KEYS ***
ODDS_API_KEYS =[
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1", "146dff42ae19a83208b059d4855095c9",
    "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9", "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b",
    "66953d96fcb2ec72a47e7f39901258e7", "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f", "1bfcac154fb2b490bf55a80a18275db1",
    "7b64d450f91079f60211cb33033594b4", "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121", "39756c093848b6968c899a4e1ded85f9", "21caebe8058c338a7ba76f6c9fb550cb", "94304f58fb95fd1d45159b9c8d292724",
    "129b9d482b270ff3809e11b0c445ef5c", "a11767c08916b29f824150bffbbf6d9c", "e7e814c95b32c4a9fb5b72acefb7389b", "3a3cd8a78555c46afa842aa8bdf1455e",
    "3b9adf0f0e27964f58de1f9b4af3adfc", "f098da95884f42852df5ff12f5704741", "768fb25c7f33328bc53b61b81e607bd9"
]

SHARP_BOOKS =["pinnacle", "betfair_ex_eu", "betonlineag", "bovada", "betus", "betrivers", "betmgm", "espnbet"]
RETAIL_BOOKS =["draftkings", "fanduel"]
ALL_BOOKS_FOR_API = list(set(SHARP_BOOKS + RETAIL_BOOKS))
ROLLING_WINDOWS =[5, 10, 20]
CONFIG = {
    "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15,
    "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20,
    "STEAM_THRESHOLD": 1.5,
    "RLM_CONFIDENCE": 0.65
}

# --- Part 2: Helper Functions ---

def run_accuracy_audit():
    print("\n" + "="*60)
    print("🧠 PHASE 0: PRE-FLIGHT ACCURACY AUDIT (Last 1,000 Games)")
    print("="*60)
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, "nba_advanced_training_data_v6.csv"))
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

        models = {
            'min': joblib.load(os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")),
            'pts': joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")),
            'reb': joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")),
            'ast': joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")),
            'fg3m': joblib.load(os.path.join(MODEL_DIR, "fg3m_lgbm_model.pkl"))
        }

        imputers = {
            'minutes': joblib.load(os.path.join(MODEL_DIR, "minutes_imputer.pkl")),
            'stats': joblib.load(os.path.join(MODEL_DIR, "stats_imputer.pkl"))
        }

        min_feats = joblib.load(os.path.join(MODEL_DIR, "minutes_feature_names.pkl"))
        stat_feats = joblib.load(os.path.join(MODEL_DIR, "stats_feature_names.pkl"))

    except Exception as e:
        print(f"❌ CRITICAL ERROR: Could not load files for audit. Error: {e}")
        return {}

    audit_df = df.sort_values('GAME_DATE', ascending=False).head(1000).copy()

    X_min = audit_df[min_feats]
    X_min_imputed = imputers['minutes'].transform(X_min)
    audit_df['PRED_MIN'] = models['min'].predict(X_min_imputed)

    audit_df['Proj_MIN'] = audit_df['PRED_MIN']
    X_stats = audit_df[stat_feats]
    X_stats_imputed = imputers['stats'].transform(X_stats)

    audit_df['PRED_PTS'] = models['pts'].predict(X_stats_imputed)
    audit_df['PRED_REB'] = models['reb'].predict(X_stats_imputed)
    audit_df['PRED_AST'] = models['ast'].predict(X_stats_imputed)
    audit_df['PRED_FG3M'] = models['fg3m'].predict(X_stats_imputed)

    # Calculate Combos
    audit_df['PRED_PRA'] = audit_df['PRED_PTS'] + audit_df['PRED_REB'] + audit_df['PRED_AST']
    audit_df['ACTUAL_PRA'] = audit_df['TARGET_PTS'] + audit_df['TARGET_REB'] + audit_df['TARGET_AST']

    audit_df['PRED_P+R'] = audit_df['PRED_PTS'] + audit_df['PRED_REB']
    audit_df['ACTUAL_P+R'] = audit_df['TARGET_PTS'] + audit_df['TARGET_REB']

    audit_df['PRED_P+A'] = audit_df['PRED_PTS'] + audit_df['PRED_AST']
    audit_df['ACTUAL_P+A'] = audit_df['TARGET_PTS'] + audit_df['TARGET_AST']

    audit_df['PRED_R+A'] = audit_df['PRED_REB'] + audit_df['PRED_AST']
    audit_df['ACTUAL_R+A'] = audit_df['TARGET_REB'] + audit_df['TARGET_AST']

    results = []
    comparison_map = {
        'PTS': ('TARGET_PTS', 'PRED_PTS'),
        'REB': ('TARGET_REB', 'PRED_REB'),
        'AST': ('TARGET_AST', 'PRED_AST'),
        '3PT': ('TARGET_FG3M', 'PRED_FG3M'),
        'PRA': ('ACTUAL_PRA', 'PRED_PRA'),
        'P+R': ('ACTUAL_P+R', 'PRED_P+R'),
        'P+A': ('ACTUAL_P+A', 'PRED_P+A'),
        'R+A': ('ACTUAL_R+A', 'PRED_R+A')
    }

    audit_hurdles = {}
    for label, (actual_col, pred_col) in comparison_map.items():
        mae = mean_absolute_error(audit_df[actual_col], audit_df[pred_col])
        rmse = np.sqrt(mean_squared_error(audit_df[actual_col], audit_df[pred_col]))
        avg_actual = audit_df[actual_col].mean()
        rel_error = (mae / avg_actual) * 100 if avg_actual > 0 else 0

        audit_hurdles[label] = rel_error

        results.append({
            'Prop Type': label,
            'Avg Error (MAE)': round(mae, 2),
            'Volatility (RMSE)': round(rmse, 2),
            'Error %': f"{round(rel_error, 1)}%"
        })

    report = pd.DataFrame(results).sort_values('Avg Error (MAE)')
    print(report.to_string(index=False))
    print("="*60 + "\n")
    return audit_hurdles

def fetch_advanced_team_defense(season):
    print("   🛡️ Fetching Advanced Team Defense (Synergy & Rim) for the entire league...")
    adv_map = {}
    team_ids = [team['id'] for team in teams.get_teams()]

    pr_df, spot_df, rim_df, perim_df = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    stype = 'Playoffs' if datetime.now().month in [4, 5, 6] else 'Regular Season'

    try:
        pr_df = synergyplaytypes.SynergyPlayTypes(play_type_nullable='PRBallHandler', type_grouping_nullable='defensive', player_or_team_abbreviation='T', season=season, season_type_all_star=stype, timeout=60).get_data_frames()[0]
        if not pr_df.empty: pr_df['PR_RANK'] = pr_df['PPP'].rank(ascending=True)
    except Exception: pass

    try:
        spot_df = synergyplaytypes.SynergyPlayTypes(play_type_nullable='Spotup', type_grouping_nullable='defensive', player_or_team_abbreviation='T', season=season, season_type_all_star=stype, timeout=60).get_data_frames()[0]
        if not spot_df.empty: spot_df['SPOT_RANK'] = spot_df['PPP'].rank(ascending=True)
    except Exception: pass

    try:
        rim_df = leaguedashptteamdefend.LeagueDashPtTeamDefend(defense_category='Less Than 6 Ft', season=season, season_type_all_star=stype, timeout=60).get_data_frames()[0]
        if not rim_df.empty: rim_df['RIM_RANK'] = rim_df['D_FG_PCT'].rank(ascending=True)
    except Exception: pass

    try:
        perim_df = leaguedashptteamdefend.LeagueDashPtTeamDefend(defense_category='Greater Than 15 Ft', season=season, season_type_all_star=stype, timeout=60).get_data_frames()[0]
        if not perim_df.empty: perim_df['PERIM_RANK'] = perim_df['D_FG_PCT'].rank(ascending=True)
    except Exception: pass

    for team_id in team_ids:
        tid = str(team_id)

        pr_val = int(pr_df.loc[pr_df['TEAM_ID'] == team_id, 'PR_RANK'].iloc[0]) if not pr_df.empty and team_id in pr_df['TEAM_ID'].values else 15
        spot_val = int(spot_df.loc[spot_df['TEAM_ID'] == team_id, 'SPOT_RANK'].iloc[0]) if not spot_df.empty and team_id in spot_df['TEAM_ID'].values else 15
        rim_val = int(rim_df.loc[rim_df['TEAM_ID'] == team_id, 'RIM_RANK'].iloc[0]) if not rim_df.empty and team_id in rim_df['TEAM_ID'].values else 15
        perim_val = int(perim_df.loc[perim_df['TEAM_ID'] == team_id, 'PERIM_RANK'].iloc[0]) if not perim_df.empty and team_id in perim_df['TEAM_ID'].values else 15

        adv_map[tid] = {
            'PR_RANK': pr_val,
            'SPOT_RANK': spot_val,
            'RIM_RANK': rim_val,
            'PERIM_RANK': perim_val
        }
    return adv_map

def fetch_league_shot_quality(season):
    print(f"   🎯 Fetching League-Wide Shot Quality Data (General & Wide Open) for {season}...")
    try:
        gen_dash = leaguedashplayerptshot.LeagueDashPlayerPtShot(
            season=season, close_def_dist_range_nullable='', per_mode_simple='PerGame', timeout=60
        )
        df_gen = gen_dash.get_data_frames()[0][['PLAYER_ID', 'EFG_PCT']]

        wo_dash = leaguedashplayerptshot.LeagueDashPlayerPtShot(
            season=season, close_def_dist_range_nullable='6+ Feet - Wide Open', per_mode_simple='PerGame', timeout=60
        )
        df_wo = wo_dash.get_data_frames()[0][['PLAYER_ID', 'FGA_FREQUENCY']]

        merged = pd.merge(df_gen, df_wo, on='PLAYER_ID', how='left').fillna(0)
        sq_map = {}
        for _, row in merged.iterrows():
            sq_map[row['PLAYER_ID']] = {
                'EFG_PCT': row['EFG_PCT'],
                'WO_FREQ': row['FGA_FREQUENCY']
            }
        return sq_map
    except Exception as e:
        print(f"   ⚠️ Error fetching Shot Quality data: {e}")
        return {}

def get_shot_quality_signal_from_map(row, sq_map):
    try:
        pid = row.get('PLAYER_ID')
        if pid not in sq_map: return "-"
        sq_data = sq_map[pid]
        wo_freq = sq_data['WO_FREQ']
        actual_efg = sq_data['EFG_PCT']
        if actual_efg == 0: return "-"
        if actual_efg > 0.62 and wo_freq < 0.15:
            return "⚠️ Luck Risk"
        elif wo_freq > 0.25:
            return "🎯 High SQ"
        elif actual_efg < 0.48 and wo_freq > 0.20:
            return "🔥 Due"
        return "-"
    except: return "-"

def normalize_player_name(name):
    if name is None: return None
    try: name = str(name)
    except: return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); return re.sub(r'\s+', ' ', name).strip()

def get_team_abbr_map(): return {team['full_name']: team['abbreviation'] for team in teams.get_teams()}

def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except: return np.nan
    return pd.to_numeric(min_str, errors='coerce')

def derive_position(row):
    pts = row.get('Pred_PTS', 0); reb = row.get('Pred_REB', 0); ast = row.get('Pred_AST', 0)
    if reb >= 8.0: return 'C'
    if reb >= 6.0 and ast < 2.5: return 'C/F'
    if ast >= 6.0: return 'PG'
    if ast >= 4.5 and reb < 4.0: return 'PG'
    if reb >= 5.0 and pts >= 12.0: return 'F'
    if pts >= 10.0 and reb < 4.5: return 'G'
    return 'G/F'

def fetch_espn_nba_injuries():
    print("\nFetching live injury report from ESPN API (NBA)...")
    try:
        teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
        teams_data = requests.get(teams_url, timeout=10).json()
        team_list = teams_data['sports'][0]['leagues'][0]['teams']
        all_injuries =[]
        for team in tqdm(team_list, desc="Fetching Team Rosters for Injuries"):
            team_id = team['team']['id']
            roster_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}?enable=roster"
            try: roster_data = requests.get(roster_url, timeout=10).json()
            except Exception: continue
            for player in roster_data.get('team', {}).get('athletes',[]):
                if 'injuries' in player and player['injuries']:
                    injury_obj = player['injuries'][0]
                    status = injury_obj.get('status', 'Unknown')
                    all_injuries.append({'Player_ESPN': player.get('displayName'), 'Status': status, 'Injury Details': injury_obj.get('date', '') + " " + injury_obj.get('type', {}).get('description', '')})

        if not all_injuries: print("   ⚠️ WARNING: No injury entries found."); return pd.DataFrame()
        injury_df = pd.DataFrame(all_injuries)
        injury_df['match_key'] = injury_df['Player_ESPN'].apply(normalize_player_name)
        print(f"   ✅ Successfully fetched {len(injury_df)} injury entries.")
        return injury_df
    except Exception as e: print(f"   ❌ FAILED to fetch ESPN injuries: {e}"); return pd.DataFrame()

def get_live_team_metrics(season):
    print(f"   📊 Fetching Team Metrics for {season}...")
    metrics_dict = {}
    try:
        metrics_df = teamestimatedmetrics.TeamEstimatedMetrics(season=season, league_id='00', timeout=60).get_data_frames()[0]
        if not metrics_df.empty:
            cols =['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
            metrics_dict = metrics_df.set_index('TEAM_ID')[cols].to_dict('index')
            return metrics_dict
    except Exception: pass

    if not metrics_dict:
        try:
            adv_stats = leaguedashteamstats.LeagueDashTeamStats(season=season, measure_type_detailed_defense='Advanced', timeout=60).get_data_frames()[0]
            if not adv_stats.empty:
                adv_stats['E_OFF_RATING'] = adv_stats['OFF_RATING']; adv_stats['E_DEF_RATING'] = adv_stats['DEF_RATING']
                adv_stats['E_NET_RATING'] = adv_stats['NET_RATING']; adv_stats['E_PACE'] = adv_stats['PACE']
                adv_stats['E_EFG_PCT'] = adv_stats['EFG_PCT']; adv_stats['E_OREB_PCT'] = adv_stats['OREB_PCT']
                cols =['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
                metrics_dict = adv_stats.set_index('TEAM_ID')[cols].to_dict('index')
        except Exception: pass
    return metrics_dict

def get_defense_ranks(season):
    print("   🛡️ Fetching Opponent Defense Rankings (Raw & Calculated)...")
    try:
        # We fetch the raw 'Opponent' stats. This is the most reliable way.
        stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            timeout=60
        ).get_data_frames()[0]

        if stats.empty: return {}, {}

        col_pts = 'OPP_PTS' if 'OPP_PTS' in stats.columns else 'PTS'
        col_reb = 'OPP_REB' if 'OPP_REB' in stats.columns else 'REB'
        col_ast = 'OPP_AST' if 'OPP_AST' in stats.columns else 'AST'
        col_3pm = 'OPP_FG3M' if 'OPP_FG3M' in stats.columns else 'FG3M'

        stats['CALC_RANK_PTS'] = stats[col_pts].rank(method='min', ascending=True)
        stats['CALC_RANK_REB'] = stats[col_reb].rank(method='min', ascending=True)
        stats['CALC_RANK_AST'] = stats[col_ast].rank(method='min', ascending=True)
        stats['CALC_RANK_FG3M'] = stats[col_3pm].rank(method='min', ascending=True)

        stats['PRA'] = stats[col_pts] + stats[col_reb] + stats[col_ast]
        stats['P+R'] = stats[col_pts] + stats[col_reb]
        stats['P+A'] = stats[col_pts] + stats[col_ast]
        stats['R+A'] = stats[col_reb] + stats[col_ast]

        stats['CALC_RANK_PRA'] = stats['PRA'].rank(method='min', ascending=True)
        stats['CALC_RANK_P+R'] = stats['P+R'].rank(method='min', ascending=True)
        stats['CALC_RANK_P+A'] = stats['P+A'].rank(method='min', ascending=True)
        stats['CALC_RANK_R+A'] = stats['R+A'].rank(method='min', ascending=True)

        ranks = {}
        raw_stats = {}

        for _, row in stats.iterrows():
            tid = str(row['TEAM_ID'])
            ranks[tid] = {
                'PTS': int(row['CALC_RANK_PTS']),
                'REB': int(row['CALC_RANK_REB']),
                'AST': int(row['CALC_RANK_AST']),
                'FG3M': int(row['CALC_RANK_FG3M']),
                'PRA': int(row['CALC_RANK_PRA']),
                'P+R': int(row['CALC_RANK_P+R']),
                'P+A': int(row['CALC_RANK_P+A']),
                'R+A': int(row['CALC_RANK_R+A'])
            }
            raw_stats[tid] = {
                'PTS': float(row.get(col_pts, 115.0)),
                'REB': float(row.get(col_reb, 44.0)),
                'AST': float(row.get(col_ast, 25.0)),
                'FG3M': float(row.get(col_3pm, 12.0))
            }
        return ranks, raw_stats
    except Exception as e:
        print(f"   ❌ FAILED to fetch defense ranks: {e}")
        return {}, {}

def get_nba_feng_ratings(season):
    print("   🧮 Calculating Feng/Massey Ratings...")
    try:
        games = leaguegamefinder.LeagueGameFinder(season_nullable=season, league_id_nullable='00', season_type_nullable='Regular Season', timeout=60).get_data_frames()[0]
        if games.empty: return {}
        games = games[games['WL'].isin(['W', 'L'])]
        games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
        team_ids = sorted(games['TEAM_ID'].unique())
        n_teams = len(team_ids)
        team_to_idx = {t: i for i, t in enumerate(team_ids)}
        M = np.zeros((n_teams, n_teams))
        p = np.zeros(n_teams)
        grouped = games.groupby('GAME_ID')
        for _, g_df in grouped:
            if len(g_df) != 2: continue
            row1 = g_df.iloc[0]; row2 = g_df.iloc[1]
            t1, t2 = row1['TEAM_ID'], row2['TEAM_ID']
            s1, s2 = row1['PTS'], row2['PTS']
            margin = s1 - s2
            i, j = team_to_idx[t1], team_to_idx[t2]
            p[i] += margin; p[j] -= margin
            M[i, i] += 1; M[j, j] += 1; M[i, j] -= 1; M[j, i] -= 1
        M[-1, :] = 1; p[-1] = 0
        ratings = np.linalg.solve(M, p)
        return {str(team_ids[i]): ratings[i] for i in range(n_teams)}
    except Exception as e:
        print(f"   ⚠️ Feng Ratings calculation failed: {e}")
        return {}

def calculate_prop_streak(player_id, prop_map, line):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return 0, 0, 0, 0
        df = pd.read_csv(gamelog_file)
        if df.empty: return 0, 0, 0, 0
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        df = df.sort_values(by='GAME_DATE', ascending=False).reset_index(drop=True)
        val_series = None
        col_name = SIMPLE_PROP_MAP.get(prop_map, prop_map)

        if col_name == 'PTS': val_series = df['PTS']
        elif col_name == 'REB': val_series = df['REB']
        elif col_name == 'AST': val_series = df['AST']
        elif col_name == 'FG3M': val_series = df['FG3M']
        elif col_name == 'PRA': val_series = df['PTS'] + df['REB'] + df['AST']
        elif col_name == 'P+R': val_series = df['PTS'] + df['REB']
        elif col_name == 'P+A': val_series = df['PTS'] + df['AST']
        elif col_name == 'R+A': val_series = df['REB'] + df['AST']
        elif col_name == 'DD': val_series = ((df['PTS']>=10).astype(int) + (df['REB']>=10).astype(int) + (df['AST']>=10).astype(int)) >= 2

        if val_series is None: return 0, 0, 0, 0

        l10_slice = val_series.head(10)

        streak_over = 0
        for val in l10_slice:
            if col_name == 'DD':
                if val: streak_over += 1
                else: break
            else:
                if val > line: streak_over += 1
                else: break

        streak_under = 0
        for val in l10_slice:
            if col_name == 'DD':
                if not val: streak_under += 1
                else: break
            else:
                if val < line: streak_under += 1
                else: break

        hits_over_l10 = l10_slice.sum() if col_name == 'DD' else (l10_slice > line).sum()
        hits_under_l10 = (10 - l10_slice.sum()) if col_name == 'DD' else (l10_slice < line).sum()

        return streak_over, streak_under, hits_over_l10, hits_under_l10
    except Exception: return 0, 0, 0, 0

def calculate_head_to_head(player_id, prop_map, line, opponent_abbr, season_year):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return "0/0"
        df = pd.read_csv(gamelog_file)
        if df.empty: return "0/0"

        # [FIX] Add Date Filter to prevent "11/72" (too many games)
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        # Only look at games from the last 2 years for relevance
        cutoff = datetime.now() - timedelta(days=365*2)
        df = df[df['GAME_DATE'] >= cutoff]

        col_name = SIMPLE_PROP_MAP.get(prop_map, prop_map)
        target_opp = str(opponent_abbr).upper()

        # Standardize common abbreviation mismatches
        abbr_map = {'PHX': 'PHO', 'PHO': 'PHX', 'GSW': 'GS', 'GS': 'GSW', 'NOP': 'NO', 'NO': 'NOP', 'SAS': 'SA', 'SA': 'SAS', 'NYK': 'NY', 'NY': 'NYK', 'UTA': 'UT', 'UT': 'UTA', 'WAS': 'WSH', 'WSH': 'WAS'}
        alt_opp = abbr_map.get(target_opp, target_opp)

        def is_matchup(m_str):
            if pd.isna(m_str): return False
            m_str = m_str.replace('.', '').upper()
            # [FIX] Strict Token Match: Matchup format is usually "TEAM vs. OPP" or "TEAM @ OPP"
            tokens = m_str.split()
            actual_opp = tokens[-1]
            return actual_opp == target_opp or actual_opp == alt_opp

        df_vs = df[df['MATCHUP'].apply(is_matchup)]
        if df_vs.empty: return "0/0"

        if col_name == 'PRA': val_series = df_vs['PTS'] + df_vs['REB'] + df_vs['AST']
        elif col_name == 'P+R': val_series = df_vs['PTS'] + df_vs['REB']
        elif col_name == 'P+A': val_series = df_vs['PTS'] + df_vs['AST']
        elif col_name == 'R+A': val_series = df_vs['REB'] + df_vs['AST']
        else: val_series = df_vs[col_name]

        hits = (val_series > line).sum()
        return f"{int(hits)}/{len(df_vs)}"
    except Exception: return "0/0"

def get_live_schedule_from_nba_data(iso_date, season_year):
    schedule = {}
    print(f"--- Fetching Schedule for {iso_date} via API (ScoreboardV3) ---")
    try:
        board = ScoreboardV3(game_date=iso_date, timeout=60)
        games = board.game_header.get_data_frame()

        if games.empty:
            print(f"Warning: NBA API returned no games for {iso_date}")
            return {}

        nba_teams = teams.get_teams()
        abbr_to_id = {t['abbreviation']: str(t['id']) for t in nba_teams}

        valid_count = 0
        for _, game in games.iterrows():
            try:
                game_code = game['gameCode'] # Format: "20260513/BOSMIA"
                teams_part = game_code.split('/')[-1]
                away_tricode = teams_part[:3]
                home_tricode = teams_part[3:]

                h_id = abbr_to_id.get(home_tricode)
                v_id = abbr_to_id.get(away_tricode)

                if h_id and v_id:
                    schedule[str(h_id)] = str(v_id)
                    schedule[str(v_id)] = str(h_id)
                    valid_count += 1
            except Exception:
                continue

        print(f"✅ Found {valid_count} valid games on the schedule.")
        return schedule

    except Exception as e:
        print(f"❌ Critical Error fetching schedule: {e}")
        return {}

def get_draft_class_ids(season_year):
    try: df_draft = drafthistory.DraftHistory(season_year_nullable=season_year, timeout=60).get_data_frames()[0]; return set(df_draft['PERSON_ID'].unique())
    except Exception: return set()
def identify_team_changes(stats_last_season, stats_current_season):
    if stats_last_season.empty or stats_current_season.empty: return set()
    merged = pd.merge(stats_last_season[['PLAYER_ID', 'TEAM_ID']], stats_current_season[['PLAYER_ID', 'TEAM_ID']], on='PLAYER_ID', suffixes=('_last', '_current'))
    return set(merged[merged['TEAM_ID_last'] != merged['TEAM_ID_current']]['PLAYER_ID'].unique())
def get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season):
    gp_last = stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id].empty else 99
    gp_current = stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id].empty else 0
    is_rookie, is_traded = player_id in rookie_ids, player_id in traded_player_ids; is_acclimated = gp_current >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else: return "Traded Veteran" if is_traded and not is_acclimated else "Low Volume" if gp_last < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"] else "Veteran"
def get_and_update_gamelogs(player_id, current_season, prev_season):
    gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
    if os.path.exists(gamelog_file):
        local_logs = pd.read_csv(gamelog_file); local_logs['GAME_DATE'] = pd.to_datetime(local_logs['GAME_DATE']); last_date = local_logs['GAME_DATE'].max()
        try:
            file_mod_time = datetime.fromtimestamp(os.path.getmtime(gamelog_file)).date()
            if file_mod_time == datetime.now().date(): return pd.read_csv(gamelog_file)
            time.sleep(0.4)
            stype = 'Playoffs' if datetime.now().month in [4, 5, 6] else 'Regular Season'
            new_logs = playergamelog.PlayerGameLog(player_id=player_id, date_from_nullable=last_date.strftime('%m/%d/%Y'), season_type_all_star=stype, timeout=30).get_data_frames()[0]
            if not new_logs.empty:
                new_logs['GAME_DATE'] = pd.to_datetime(new_logs['GAME_DATE']); new_logs = new_logs[new_logs['GAME_DATE'] > last_date]
                if not new_logs.empty: updated_logs = pd.concat([new_logs, local_logs], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); updated_logs.to_csv(gamelog_file, index=False); return updated_logs
            return local_logs
        except Exception: return local_logs
    else:
        try:
            time.sleep(0.4)
            cur = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, timeout=30).get_data_frames()[0]
            prev = playergamelog.PlayerGameLog(player_id=player_id, season=prev_season, timeout=30).get_data_frames()[0]
            full_logs = pd.concat([cur, prev], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); full_logs.to_csv(gamelog_file, index=False); return full_logs
        except Exception: return pd.DataFrame()

def calculate_live_features(logs):
    if logs.empty or len(logs) < 5: return {}
    logs_for_calc = logs.copy(); features = {}
    logs_for_calc['MIN'] = logs_for_calc['MIN'].apply(convert_min_to_decimal)
    base_stats =['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FG3M', 'FTA', 'TOV', 'FGM']
    for col in base_stats:
        if col in logs_for_calc.columns: logs_for_calc[col] = pd.to_numeric(logs_for_calc[col], errors='coerce')
        else: logs_for_calc[col] = np.nan
    logs_for_calc.dropna(subset=['MIN'], inplace=True);
    if logs_for_calc.empty: return {}
    avg_minutes = logs_for_calc['MIN'].mean()
    weights = np.ones(len(logs_for_calc)); weights[logs_for_calc['MIN'] < 0.7 * avg_minutes] = 0.5; weights[logs_for_calc['MIN'] > 1.15 * avg_minutes] = 1.2
    logs_for_calc['WEIGHT'] = weights
    logs_safe = logs_for_calc[logs_for_calc['MIN'] > 0].copy()

    logs_safe['EFG_PCT_RAW'] = (logs_safe['FGM'] + 0.5 * logs_safe['FG3M']) / logs_safe['FGA'].replace(0, np.nan)
    logs_safe['TS_PCT_RAW'] = logs_safe['PTS'] / (2 * (logs_safe['FGA'] + 0.44 * logs_safe['FTA'])).replace(0, np.nan)
    logs_safe[['EFG_PCT_RAW', 'TS_PCT_RAW']] = logs_safe[['EFG_PCT_RAW', 'TS_PCT_RAW']].fillna(0)

    # Efficiency Feature Alignment for new models
    features['EFG_PCT_PREV'] = logs_safe['EFG_PCT_RAW'].iloc[0] if not logs_safe.empty else 0
    features['TS_PCT_PREV'] = logs_safe['TS_PCT_RAW'].iloc[0] if not logs_safe.empty else 0

    stats_for_ratios =['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FTA', 'TOV', 'FG3M']
    for stat in stats_for_ratios:
        if stat in logs_safe.columns: logs_safe[f'{stat}_PM'] = logs_safe[stat] / logs_safe['MIN']
        else: logs_safe[f'{stat}_PM'] = 0

    stats_to_roll =['PTS', 'REB', 'AST', 'MIN', 'FG3M', 'FG3A', 'EFG_PCT_RAW', 'TS_PCT_RAW'] +[f'{s}_PM' for s in stats_for_ratios]
    for stat in stats_to_roll:
        if stat not in logs_safe.columns: continue
        for w in ROLLING_WINDOWS:
            recent_logs = logs_safe.head(w)
            features[f'W_ROLL_AVG_{stat}_{w}'] = np.average(recent_logs[stat].fillna(0), weights=recent_logs['WEIGHT'].head(len(recent_logs))) if not recent_logs.empty else np.nan
        features[f'TREND_{stat}'] = features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[0]}') - features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[-1]}')

    for stat in['PTS', 'REB', 'AST', 'FG3M']:
        if stat in logs_safe.columns:
            last_20 = logs_safe.head(20)
            if not last_20.empty:
                std_dev = last_20[stat].std(); mean_val = last_20[stat].mean()
                cv = (std_dev / mean_val) if mean_val > 0.5 else 9.9
                features[f'CV_{stat}'] = cv
            else: features[f'CV_{stat}'] = 9.9
    features['DAYS_OF_REST'] = (datetime.utcnow().replace(tzinfo=pytz.utc) - pd.to_datetime(logs_for_calc['GAME_DATE'].iloc[0]).tz_localize('UTC')).days
    for stat in stats_for_ratios: features[f'{stat}_PM'] = logs_safe.head(20)[f'{stat}_PM'].mean() if f'{stat}_PM' in logs_safe.columns else 0
    return features

def calculate_ou_probability(row):
    try:
        projection = float(row.get('Projection', 0))
        line = float(row.get('Line', 0))
        prop = row.get('Prop')

        # Binary Probabilities for DD/TD
        if prop in['DD', 'TD']:
            prob_over = projection / 100.0
            prob_under = 1.0 - prob_over
            return prob_over, prob_under

        # Poisson for Counting Stats
        if pd.isna(projection) or pd.isna(line) or projection <= 0: return 0.5, 0.5
        prob_over = 1 - poisson.cdf(k=line, mu=projection)
        prob_under = poisson.cdf(k=line - 1, mu=projection)
        return prob_over, prob_under
    except: return 0.5, 0.5

def get_previous_run_data(current_date_str, output_dir):
    try:
        dt_obj = datetime.strptime(current_date_str, '%m-%d-%Y')
        short_date_pattern = f"{dt_obj.month}-{dt_obj.day}-{str(dt_obj.year)[-2:]}"
    except:
        short_date_pattern = None

    candidate_files = []
    if os.path.exists(output_dir):
        for f in os.listdir(output_dir):
            if f.endswith("NBA Pro Dashboard.xlsx"):
                if current_date_str in f or (short_date_pattern and f.startswith(short_date_pattern)):
                    candidate_files.append(os.path.join(output_dir, f))

    if candidate_files:
        candidate_files.sort(key=os.path.getmtime, reverse=True)
        for file_path in candidate_files:
            print(f"✅ Found earlier run from TODAY: {os.path.basename(file_path)}. Calculating Pro vs Info Window movement...")
            try:
                return pd.read_excel(file_path, sheet_name='DraftKings_EV_Bets')
            except Exception:
                continue

    print(f"ℹ No earlier run found for TODAY ({current_date_str}). This is likely the 'Pro Window' (First Run).")
    return pd.DataFrame()

def apply_movement_and_ranking(curr_df, prev_df, sharp_props_map):
    if curr_df.empty: return curr_df
    if not prev_df.empty:
        curr_df['Player'] = curr_df['Player'].astype(str); prev_df['Player'] = prev_df['Player'].astype(str)
        if 'Bookmaker' in prev_df.columns:
            curr_df['Bookmaker'] = curr_df['Bookmaker'].astype(str); prev_df['Bookmaker'] = prev_df['Bookmaker'].astype(str)
            prev_clean = prev_df[['Player', 'Prop', 'Bookmaker', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop', 'Bookmaker'], how='left', suffixes=('', '_prev'))
        else:
            prev_clean = prev_df[['Player', 'Prop', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop'], how='left', suffixes=('', '_prev'))
    else: merged = curr_df.copy(); merged['Prev_Edge'] = np.nan

    def get_sharp_data(row):
        key = (row['match_key'], row['Prop'])
        sharp_lines_list = sharp_props_map.get(key,[])
        if sharp_lines_list:
            avg_sharp_line = sum(sharp_lines_list) / len(sharp_lines_list)
            diff = row['Projection'] - avg_sharp_line
            return round(avg_sharp_line, 1), round(diff, 1)
        return np.nan, np.nan
    merged[['Sharp Line', 'Proj vs Sharp']] = merged.apply(lambda r: pd.Series(get_sharp_data(r)), axis=1)

    def get_signal(row):
        curr_edge, prev_edge = row.get('Edge (%)', 0), row.get('Prev_Edge')
        if pd.isna(prev_edge): return "-"
        delta = curr_edge - prev_edge
        if delta > 3.5: return "PRICE IMPROVED"
        if delta < -3.5 and curr_edge > 5: return "SHARPS AGREE"
        return "-"
    merged['Sharp Signal'] = merged.apply(get_signal, axis=1)

    def get_consistency_badge(row):
        cv_pts = row.get('CV_PTS', 9.9); cv_reb = row.get('CV_REB', 9.9); cv_ast = row.get('CV_AST', 9.9)
        prop = row['Prop']
        cv = 9.9
        if prop == 'PTS': cv = cv_pts
        elif prop == 'REB': cv = cv_reb
        elif prop == 'AST': cv = cv_ast
        elif prop in['FG3M', '3PT']: cv = row.get('CV_FG3M', 9.9)
        elif prop == 'PRA': cv = (cv_pts + cv_reb + cv_ast) / 3
        elif prop == 'P+R': cv = (cv_pts + cv_reb) / 2
        elif prop == 'P+A': cv = (cv_pts + cv_ast) / 2
        elif prop == 'R+A': cv = (cv_reb + cv_ast) / 2
        else: return "-"
        if cv > 5.0: return "-"
        if cv < 0.25: return "🔒 High"
        elif cv < 0.45: return "✅ Med"
        else: return "⚡ Volatile"
    merged['Consistency'] = merged.apply(get_consistency_badge, axis=1)

    def get_ranking(row):
        edge, signal = row.get('Edge (%)', 0), row['Sharp Signal']
        bet_type = row.get('Bet')
        sl, line = row.get('Sharp Line'), row.get('Line')
        if not pd.isna(sl):
            if (bet_type == 'Over' and sl > line) or (bet_type == 'Under' and sl < line):
                return "💎 SHARP ALIGNMENT"
        if edge > 15.0: return "Strong Value"
        if edge > 5.0: return "Solid Play"
        return "-"
    merged['Ranking'] = merged.apply(get_ranking, axis=1)
    return merged

# --- Part 5: Main Execution Workflow ---
def run_daily_projections():
    proj_df = pd.DataFrame()

    try:
        if not os.path.exists('/content/drive'): drive.mount('/content/drive', force_remount=True)
    except Exception as e: print(f"⚠️ Drive mount check skipped: {e}")

    # [NEW] Phase 0: Self-Correction via Accuracy Audit Script
    audit_hurdles = run_accuracy_audit()

    eastern = pytz.timezone('US/Eastern')
    SIMULATION_DATE = None

    if 'SIMULATION_DATE' in globals() and globals()['SIMULATION_DATE'] is not None:
         try: datetime.strptime(globals()['SIMULATION_DATE'], "%Y-%m-%d"); SIMULATION_DATE = globals()['SIMULATION_DATE']
         except: SIMULATION_DATE = None

    if SIMULATION_DATE:
        live_now_et = datetime.strptime(f"{SIMULATION_DATE} 12:00:00", "%Y-%m-%d %H:%M:%S").astimezone(eastern)
        print(f"\n⚠️ RUNNING IN SIMULATION MODE for date: {SIMULATION_DATE}")
        iso_today_str = SIMULATION_DATE
        file_today_str = live_now_et.strftime('%m-%d-%Y')
    else:
        live_now_et = datetime.now(eastern)
        print(f"\n📅 Running Live Mode. Current ET Time: {live_now_et.strftime('%Y-%m-%d %H:%M:%S')}")
        iso_today_str = live_now_et.strftime('%Y-%m-%d')
        file_today_str = live_now_et.strftime('%m-%d-%Y')

    run_mode_input = ''
    while run_mode_input not in['i', 'f', 't']:
        run_mode_input = input("Enter run mode ('i' for initial capture, 'f' for final report, 't' for training): ").strip().lower()

    if run_mode_input == 'i':
        run_mode = 'initial'
        print("\n🔵 INITIAL RUN DETECTED. Will save opening lines to CSV.")
    elif run_mode_input == 'f':
        run_mode = 'final'
        print("\n🔴 FINAL RUN DETECTED. Will compare vs Initial Lines for Steam/RLM/CLV.")
    elif run_mode_input == 't':
        run_mode = 'training'
        print("\n🟢 TRAINING MODE DETECTED.")

    if live_now_et.month >= 9: start_year = live_now_et.year; end_year = live_now_et.year + 1
    else: start_year = live_now_et.year - 1; end_year = live_now_et.year
    season = f"{start_year}-{str(end_year)[-2:]}"; prev_season = f"{start_year-1}-{str(start_year)[-2:]}"; season_year_for_url = str(start_year)

    print(f"Running projections for {iso_today_str} (Season: {season})")

    try:
        models = {'minutes': joblib.load(os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")), 'pts': joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")), 'reb': joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")), 'ast': joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")), 'fg3m': joblib.load(os.path.join(MODEL_DIR, "fg3m_lgbm_model.pkl")), 'dd': joblib.load(os.path.join(MODEL_DIR, "dd_lgbm_model.pkl")), 'td': joblib.load(os.path.join(MODEL_DIR, "td_lgbm_model.pkl"))}
        imputers = {name: joblib.load(os.path.join(MODEL_DIR, f"{name}_imputer.pkl")) for name in['minutes', 'stats']}
        min_feature_names = joblib.load(os.path.join(MODEL_DIR, "minutes_feature_names.pkl")); stat_feature_names = joblib.load(os.path.join(MODEL_DIR, "stats_feature_names.pkl")); mae_values = json.load(open(os.path.join(MODEL_DIR, "mae_values.json")))
        print("✅ All models, imputers, and MAE values loaded successfully.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: A required model file is missing. Please run Script 2. Details: {e}"); return

    injury_df = fetch_espn_nba_injuries()

    position_map = {}
    POSITIONS_FILE = os.path.join(GDRIVE_BASE_PATH, "Player Positions/2025-26 NBA Player Positions.xlsx")
    if os.path.exists(POSITIONS_FILE):
        try:
            pos_df = pd.read_excel(POSITIONS_FILE)
            if 'Player' in pos_df.columns and 'Pos' in pos_df.columns:
                pos_df['match_key'] = pos_df['Player'].apply(normalize_player_name)
                position_map = pos_df.set_index('match_key')['Pos'].to_dict()
        except Exception: pass

    scratched_players =[]
    if not injury_df.empty:
        scratched_players = injury_df[injury_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]['match_key'].tolist()

    if scratched_players: print(f"   ℹ Detected {len(scratched_players)} ruled-out players. Adjusting team usage automatically...")

    print("\n--- Preparing Context & Stats ---")
    stats_current_season, stats_last_season = pd.DataFrame(), pd.DataFrame()
    for attempt in range(3):
        try:
            # [FIX] Combine Regular Season and Playoffs stats for May Postgraduate rosters
            stats_current_season = leaguedashplayerstats.LeagueDashPlayerStats(season=season, season_type_all_star='Regular Season', timeout=120).get_data_frames()[0]
            try:
                stats_playoffs = leaguedashplayerstats.LeagueDashPlayerStats(season=season, season_type_all_star='Playoffs', timeout=120).get_data_frames()[0]
                if not stats_playoffs.empty:
                    stats_current_season = pd.concat([stats_current_season, stats_playoffs]).drop_duplicates(subset=['PLAYER_ID'])
            except: pass

            # [FIXED TYPE BUG] Corrected endpoint call to retrieve Player Stats instead of Team Stats
            stats_last_season = leaguedashplayerstats.LeagueDashPlayerStats(season=prev_season, timeout=120).get_data_frames()[0]
            if not stats_current_season.empty and not stats_last_season.empty: print("✅ Successfully fetched season stats."); break
        except Exception: time.sleep(5)

    def_ranks, def_stats_raw = get_defense_ranks(season)
    adv_def_map = fetch_advanced_team_defense(season)
    feng_ratings = get_nba_feng_ratings(season)
    team_metrics = get_live_team_metrics(season)
    schedule = get_live_schedule_from_nba_data(iso_today_str, season_year_for_url)

    if schedule:
        schedule = {str(k): str(v) for k, v in schedule.items() if k is not None and v is not None}
    if not stats_current_season.empty: stats_current_season['TEAM_ID'] = stats_current_season['TEAM_ID'].astype(str).str.split('.').str[0]

    rookie_ids = get_draft_class_ids(season_year_for_url); traded_player_ids = identify_team_changes(stats_last_season, stats_current_season)
    props_df, live_game_lines = pd.DataFrame(), {}
    key, game_events = None,[]
    team_abbr_map = get_team_abbr_map()
    sq_map = fetch_league_shot_quality(season)

    print("\n--- Running Initial Projections & Monte Carlo Sims ---")
    all_rosters = stats_current_season
    projection_rows, model_game_rows = [],[]

    LEAGUE_AVG_PACE = 99.0
    LEAGUE_AVG_OFF = 115.0

    if team_metrics:
        paces = [m['E_PACE'] for m in team_metrics.values() if m['E_PACE'] > 0]
        offs =[m['E_OFF_RATING'] for m in team_metrics.values() if m['E_OFF_RATING'] > 0]
        if paces: LEAGUE_AVG_PACE = sum(paces) / len(paces)
        if offs: LEAGUE_AVG_OFF = sum(offs) / len(offs)

    if schedule:
        for team_id, opponent_id in tqdm(schedule.items(), desc="Generating Projections"):
            roster = all_rosters[all_rosters['TEAM_ID'] == str(team_id)]
            team_scratches = []
            if not roster.empty:
                team_scratches =[p for p in scratched_players if p in roster['PLAYER_NAME'].apply(normalize_player_name).tolist()]
            usage_boost_factor = 1.0 + (len(team_scratches) * 0.05)

            for _, row in roster.iterrows():
                if 'GP' in row and row['GP'] > 0 and (row['MIN'] / row['GP']) < 6.0: continue
                player_id, player_name, team_abbr = row['PLAYER_ID'], row['PLAYER_NAME'], row['TEAM_ABBREVIATION']
                match_key = normalize_player_name(player_name)
                logs = get_and_update_gamelogs(player_id, season, prev_season); features = calculate_live_features(logs)
                if not features: continue
                try: opp_id_int = int(opponent_id); opp_metrics = team_metrics.get(opp_id_int, {})
                except: opp_metrics = {}

                features['OPP_ALLOWED_PTS_AVG_30'] = opp_metrics.get('E_DEF_RATING', 115.0)
                features['OPP_ALLOWED_REB_AVG_30'] = 44.0; features['OPP_ALLOWED_AST_AVG_30'] = 25.0; features['OPP_ALLOWED_FG3M_AVG_30'] = 12.0
                features['OPP_ALLOWED_EFG_PCT_AVG_30'] = opp_metrics.get('E_EFG_PCT', 0.54)

                if str(opponent_id) in def_stats_raw:
                    features['OPP_ALLOWED_PTS_AVG_30'] = def_stats_raw[str(opponent_id)]['PTS']
                    features['OPP_ALLOWED_REB_AVG_30'] = def_stats_raw[str(opponent_id)]['REB']
                    features['OPP_ALLOWED_AST_AVG_30'] = def_stats_raw[str(opponent_id)]['AST']
                    features['OPP_ALLOWED_FG3M_AVG_30'] = def_stats_raw[str(opponent_id)]['FG3M']

                features['VEGAS_SPREAD'] = 0; features['VEGAS_TOTAL'] = 225

                try:
                    # [ALIGNMENT FIX] Use Reindex to prevent Mismatch crashes
                    min_input = pd.DataFrame([features]).reindex(columns=min_feature_names, fill_value=0)
                    pred_min = models['minutes'].predict(imputers['minutes'].transform(min_input))[0]; features['Proj_MIN'] = pred_min

                    if usage_boost_factor > 1.0:
                        features['FGA_PM'] = features.get('FGA_PM', 0) * usage_boost_factor
                        pred_min = pred_min * 1.02

                    stat_input_df = pd.DataFrame([features]).reindex(columns=stat_feature_names, fill_value=0)
                    stat_input = imputers['stats'].transform(stat_input_df)

                    p_pts = models['pts'].predict(stat_input)[0]; p_reb = models['reb'].predict(stat_input)[0]
                    p_ast = models['ast'].predict(stat_input)[0]; p_3pm = models['fg3m'].predict(stat_input)[0]

                    temp_proj = {'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast}
                    if match_key in position_map: player_pos = position_map[match_key]
                    else: player_pos = derive_position(temp_proj)

                    opp_team_data = teams.find_team_name_by_id(int(opponent_id))
                    opp_abbr = opp_team_data['abbreviation'] if opp_team_data else "UNK"

                    sq_signal = get_shot_quality_signal_from_map(features, sq_map)

                    proj = {
                        'PLAYER_ID': player_id, 'match_key': match_key, 'Player': player_name,
                        'Pos': player_pos, 'Experience': get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season),
                        'Team': team_abbr, 'Opponent': opp_abbr,
                        'OpponentID': str(opponent_id), 'Pred_MIN': pred_min, 'Pred_FGA_PM': features.get('FGA_PM', 0),
                        'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast, 'Pred_FG3M': p_3pm, 'EFG_PCT_PREV': features.get('EFG_PCT_PREV', 0), 'TEAM_ID': team_id,
                        'Shot Quality': sq_signal, 'TREND_PTS': features.get('TREND_PTS', 0)
                    }
                    # [NEW] Star Power Identification
                    proj['Is_Star'] = (p_pts >= 22) or (pred_min >= 34)
                    proj['CV_PTS'] = features.get('CV_PTS', 9.9); proj['CV_REB'] = features.get('CV_REB', 9.9)
                    proj['CV_AST'] = features.get('CV_AST', 9.9); proj['CV_FG3M'] = features.get('CV_FG3M', 9.9)
                    clf_features = np.array([proj['Pred_PTS'], proj['Pred_REB'], proj['Pred_AST']]).reshape(1, -1)
                    proj['Pred_DD'] = models['dd'].predict_proba(clf_features)[0][1] * 100; proj['Pred_TD'] = models['td'].predict_proba(clf_features)[0][1] * 100
                    proj['Pred_PRA'] = proj['Pred_PTS'] + proj['Pred_REB'] + proj['Pred_AST']; proj['Pred_P+R'] = proj['Pred_PTS'] + proj['Pred_REB']; proj['Pred_P+A'] = proj['Pred_PTS'] + proj['Pred_AST']; proj['Pred_R+A'] = proj['Pred_REB'] + proj['Pred_AST']
                    projection_rows.append(proj)
                except Exception: continue

            home_stats, away_stats = team_metrics.get(int(team_id), {}), team_metrics.get(int(opponent_id), {})
            if home_stats and away_stats:
                h_off, a_off = home_stats.get('E_OFF_RATING', 110), away_stats.get('E_OFF_RATING', 110)
                h_def, a_def = home_stats.get('E_DEF_RATING', 110), away_stats.get('E_DEF_RATING', 110)
                h_pace, a_pace = home_stats.get('E_PACE', 99), away_stats.get('E_PACE', 99)
                poss = (h_pace + a_pace) / 2; HCA = 2.5

                # [NEW] Dynamic HCA (MSG Factor)
                high_intensity_venues = ['New York Knicks', 'Boston Celtics', 'Philadelphia 76ers', 'Denver Nuggets']
                home_team_name = teams.find_team_name_by_id(int(team_id))['full_name']
                if home_team_name in high_intensity_venues:
                    HCA = 4.5 # Atmosphere Boost

                h_exp = ((h_off * a_def) / LEAGUE_AVG_OFF) * (poss / 100.0) + (HCA/2)
                a_exp = ((a_off * h_def) / LEAGUE_AVG_OFF) * (poss / 100.0) - (HCA/2)
                SIMS = 5000
                scores = np.random.multivariate_normal([h_exp, a_exp], [[169, 50],[50, 169]], size=SIMS)
                h_sc, a_sc = scores[:,0], scores[:,1]
                model_spread = -(np.mean(h_sc) - np.mean(a_sc))
                model_total = np.mean(h_sc) + np.mean(a_sc)
                model_win_prob = np.mean(h_sc > a_sc)
                pyth_h = h_off**13.91 / (h_off**13.91 + h_def**13.91); pyth_a = a_off**13.91 / (a_off**13.91 + a_def**13.91)
                log5_prob = (pyth_h - pyth_h * pyth_a) / (pyth_h + pyth_a - 2 * pyth_h * pyth_a)
                actual_win_pct = home_stats.get('W', 0) / home_stats.get('GP', 1)
                luck_metric = actual_win_pct - pyth_h
                r_home, r_away = feng_ratings.get(str(team_id), 0), feng_ratings.get(str(opponent_id), 0)
                feng_spread = -(r_home - r_away + HCA) if r_home and r_away else "-"
                model_game_rows.append({'TeamID': int(team_id), 'Home_Team_Name': home_team_name, 'Model Spread': round(model_spread, 1), 'Model Total': round(model_total, 1), 'Model Win %': model_win_prob, 'Log5 %': log5_prob, 'Feng Spread': round(feng_spread, 1) if isinstance(feng_spread, float) else feng_spread, 'Luck': luck_metric, 'NetRtg': home_stats.get('E_NET_RATING'), 'eFG%': home_stats.get('E_EFG_PCT'), 'OREB%': home_stats.get('E_OREB_PCT')})

    proj_df = pd.DataFrame(projection_rows)
    if proj_df.empty: print(f"\n❌ NO PROJECTIONS GENERATED."); return
    else: proj_df['match_key'] = proj_df['Player'].apply(normalize_player_name); print(f"✅ Generated {len(proj_df)} player projections.")

    # --- Market Odds Fetching (REVERTED TO v121 LOGIC) ---
    print("\n--- Fetching Market Odds Data ---")
    game_lines_rows =[]; sharp_props_map = {}; initial_lines_to_save =[]
    all_props_raw =[]

    # Loop through keys to find one with quota
    key = None
    for k in ODDS_API_KEYS:
        try:
            r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds?apiKey={k}&regions=us&markets=h2h", timeout=10)
            if r.status_code == 200:
                remaining = int(r.headers.get('x-requests-remaining', 0))
                if remaining > 10:
                    key = k; break
        except: continue

    if key:
        try:
            # [REVERT] Back to v121 style fetch without commenceTime filters
            url_games = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey={key}&regions=us,eu&markets=h2h,spreads,totals&bookmakers={','.join(ALL_BOOKS_FOR_API)}"
            games_data = requests.get(url_games, timeout=15).json()

            # [FIX] Ensure games_data is a list to prevent TypeError
            if isinstance(games_data, list):
                for game in games_data:
                    # Filter for today's games in Python
                    if parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%Y-%m-%d') != iso_today_str:
                        continue

                    home_abbr = team_abbr_map.get(game['home_team'], game['home_team'])
                    away_abbr = team_abbr_map.get(game['away_team'], game['away_team'])

                    # Find matching model data
                    model_data = next((item for item in model_game_rows if item["Home_Team_Name"] == game['home_team']), {})

                    spreads, totals = [], []
                    retail_spread, retail_total, retail_ml = np.nan, np.nan, np.nan

                    # [FIXED SEQUENCE] Parse bookmakers and calculate sharp spreads BEFORE executing CLV logic check
                    for book in game.get('bookmakers',[]):
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads':
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']:
                                        if book['key'] in SHARP_BOOKS: spreads.append(o['point'])
                                        if book['key'] in RETAIL_BOOKS: retail_spread = o['point']
                            elif market['key'] == 'totals':
                                for o in market['outcomes']:
                                    if o['name'] == 'Over':
                                        if book['key'] in SHARP_BOOKS: totals.append(o['point'])
                                        if book['key'] in RETAIL_BOOKS: retail_total = o['point']
                            elif market['key'] == 'h2h':
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team'] and book['key'] in RETAIL_BOOKS: retail_ml = o['price']

                    sharp_spread = sum(spreads)/len(spreads) if spreads else None
                    sharp_total = sum(totals)/len(totals) if totals else None

                    # [FIX] CLV/Steam Logic population (Now resolves safely without scoping errors)
                    mvmt, clv_s, clv_t = "-", "-", "-"
                    if run_mode == 'final' and os.path.exists(INITIAL_LINES_PATH):
                        try:
                            init_df = pd.read_csv(INITIAL_LINES_PATH); matched = init_df[init_df['GameID'] == game['id']]
                            if not matched.empty:
                                init_val = matched.iloc[0]['Initial_Sharp_Spread']
                                if sharp_spread is not None:
                                    clv_s = round(init_val - sharp_spread, 1)
                                    if abs(clv_s) >= 1.5: mvmt = "🔥 Steam Chase"
                                else:
                                    clv_s = 0.0
                        except: pass

                    game_lines_rows.append({
                        'Game': f"{away_abbr} @ {home_abbr}", 'Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'),
                        'Retail Spread': retail_spread, 'Sharp Spread': round(sharp_spread, 1) if sharp_spread else "-",
                        'Model Spread': model_data.get('Model Spread', '-'), 'Retail Total': retail_total,
                        'Sharp Total': round(sharp_total, 1) if sharp_total else "-", 'Model Total': model_data.get('Model Total', '-'),
                        'Retail ML': retail_ml, 'Model Win %': f"{model_data.get('Model Win %', 0)*100:.1f}%" if 'Model Win %' in model_data else "-",
                        'Home_Team_Full': game['home_team'], 'Sort_Time': game['commence_time'], 'Game Move': mvmt, 'Game CLV': clv_s
                    })

                    if run_mode == 'initial':
                        initial_lines_to_save.append({
                            'GameID': game['id'], 'Home': home_abbr, 'Away': away_abbr,
                            'Initial_Sharp_Spread': sharp_spread, 'Initial_Sharp_Total': sharp_total
                        })

                    # Fetch Props for this specific event
                    url_props = f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{game['id']}/odds?apiKey={key}&regions=us,eu&markets={','.join(ALL_PROP_MARKETS_LIST)}&bookmakers={','.join(ALL_BOOKS_FOR_API)}"
                    try:
                        time.sleep(0.5)
                        payload = requests.get(url_props, timeout=20).json()
                        if isinstance(payload, dict) and 'bookmakers' in payload:
                            for book in payload.get('bookmakers',[]):
                                for market in book.get('markets',[]):
                                    prop_name = MARKET_NAME_MAP.get(market['key'])
                                    if not prop_name: continue
                                    for outcome in market.get('outcomes',[]):
                                        p_name = outcome.get('description')
                                        if p_name:
                                            norm_name = normalize_player_name(p_name)
                                            if book['key'] in SHARP_BOOKS and outcome.get('point'):
                                                sharp_props_map.setdefault((norm_name, prop_name),[]).append(outcome.get('point'))
                                            if book['key'] in RETAIL_BOOKS:
                                                all_props_raw.append({
                                                    'Game': f"{away_abbr} @ {home_abbr}", 'Game_Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'),
                                                    'Player': p_name, 'Prop': prop_name, 'Line': outcome.get('point'), 'Decimal_Odds': outcome.get('price'),
                                                    'OutcomeName': outcome['name'], 'Bookmaker': book['title'], 'Home_Team_Full': game['home_team'], 'Opponent_Abbr': away_abbr if game['home_team'] == game['home_team'] else home_abbr, 'Sort_Time': game['commence_time'],
                                                    'Game Move': mvmt, 'Game CLV': clv_s
                                                })
                    except: continue
            else:
                print(f"⚠️ API Error or No Games: {games_data}")
        except: traceback.print_exc()

    if run_mode == 'initial':
        try: pd.DataFrame(initial_lines_to_save).to_csv(INITIAL_LINES_PATH, index=False)
        except: pass

    if not all_props_raw:
        print("No props found. This could be due to API issues or no games on the schedule.")
        return

    props_df = pd.DataFrame(all_props_raw)
    props_df['match_key'] = props_df['Player'].apply(normalize_player_name)
    ou_pivot = props_df[props_df['OutcomeName'].isin(['Over', 'Under'])].pivot_table(index=['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time', 'Home_Team_Full', 'Opponent_Abbr', 'Game Move', 'Game CLV'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Over': 'Over_Odds', 'Under': 'Under_Odds'})
    merged = pd.merge(ou_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key')

    merged['Status'] = 'Active'

    def get_display_odds(row):
        bet_type = row.get('Bet', 'Pass')
        dec = row.get('Over_Odds', 0) if bet_type == 'Over' else row.get('Under_Odds', 0)
        if pd.isna(dec) or dec == 0: return ""
        return f"+{int((dec - 1) * 100)}" if dec >= 2.00 else f"{int(-100 / (dec - 1))}"

    def get_proj(row):
        prop = row['Prop']
        if prop == 'PTS': return row.get('Pred_PTS', 0)
        elif prop == 'REB': return row.get('Pred_REB', 0)
        elif prop == 'AST': return row.get('Pred_AST', 0)
        elif prop == '3PT': return row.get('Pred_FG3M', 0)
        elif prop == 'PRA': return row.get('Pred_PRA', 0)
        elif prop == 'P+R': return row.get('Pred_P+R', 0)
        elif prop == 'P+A': return row.get('Pred_P+A', 0)
        elif prop == 'R+A': return row.get('Pred_R+A', 0)
        elif prop == 'DD': return row.get('Pred_DD', 0)
        elif prop == 'TD': return row.get('Pred_TD', 0)
        return 0

    merged['Projection'] = merged.apply(get_proj, axis=1).round(1)
    merged['Proj Range'] = merged.apply(lambda r: f"{max(0, r['Projection'] - mae_values.get(SIMPLE_PROP_MAP.get(r['Prop'], 'PTS'), 1)):.1f}-{r['Projection'] + mae_values.get(SIMPLE_PROP_MAP.get(r['Prop'], 'PTS'), 1):.1f}", axis=1)

    def get_opp_rank(row):
        try: oid = str(int(float(row['OpponentID'])))
        except: return "-"
        rank = def_ranks.get(oid, {}).get(SIMPLE_PROP_MAP.get(row['Prop'], 'PTS'), 99)
        if rank == 99: return "-"
        if rank <= 5: return f"{rank} (Hard)"
        if rank >= 25: return f"{rank} (Easy)"
        return str(rank)

    merged['Opp Rank'] = merged.apply(get_opp_rank, axis=1)
    merged[['Prob_Over', 'Prob_Under']] = merged.apply(lambda r: calculate_ou_probability(r), axis=1, result_type='expand')
    merged['Bet'] = np.where(merged['Prob_Over'] > 0.55, 'Over', np.where(merged['Prob_Under'] > 0.55, 'Under', 'Pass'))
    merged['Edge (%)'] = ((np.where(merged['Bet'] == 'Over', merged['Prob_Over']*merged['Over_Odds'], merged['Prob_Under']*merged['Under_Odds']) - 1) * 100).astype(float).round(1)
    merged['Odds'] = merged.apply(get_display_odds, axis=1)

    # --- SMART STREAK LOGIC ---
    def get_streak_and_l10(row):
        so, su, ho, hu = calculate_prop_streak(row['PLAYER_ID'], SIMPLE_PROP_MAP.get(row['Prop'], 'PTS'), row['Line'])
        if row['Bet'] == 'Over':
            return f"W{so}", f"{int(ho)}/10"
        else:
            return f"W{su}", f"{int(hu)}/10"

    streak_l10_df = merged.apply(get_streak_and_l10, axis=1, result_type='expand')
    merged['Streak'] = streak_l10_df[0]
    merged['L10'] = streak_l10_df[1]

    # [FIX] Robust H2H Calculation
    merged['vs Opp'] = merged.apply(lambda r: calculate_head_to_head(r['PLAYER_ID'], r['Prop'], r['Line'], r['Opponent_Abbr'], start_year), axis=1)

    prev_run_df = get_previous_run_data(file_today_str, OUTPUT_DIR)
    merged = apply_movement_and_ranking(merged, prev_run_df, sharp_props_map)

    # [FIX] Explicit population of Market Execution Columns to prevent blank cells
    merged['Market Disc.'] = (merged['Sharp Line'] - merged['Line']).round(1)
    merged['Model Alpha'] = (merged['Projection'] - merged['Line']).round(1)

    merged['Shot Quality'] = merged.apply(lambda row: get_shot_quality_signal_from_map(row, sq_map), axis=1)

    def get_adv_matchup_signal(row):
        try: adv_stats = adv_def_map.get(str(int(float(row.get('OpponentID', 0)))), {})
        except: return "-"
        if not adv_stats: return "-"

        pr_rank = adv_stats.get('PR_RANK', 15); spot_rank = adv_stats.get('SPOT_RANK', 15)
        rim_rank = adv_stats.get('RIM_RANK', 15); perim_rank = adv_stats.get('PERIM_RANK', 15)
        pos = str(row.get('Pos', '')); prop_type = str(row.get('Prop', '')); bet_type = str(row.get('Bet', ''))

        is_handler = 'PG' in pos or float(row.get('Pred_AST', 0)) >= 4.5
        is_big = 'C' in pos or float(row.get('Pred_REB', 0)) >= 6.5

        if bet_type == 'Over':
            if is_handler and prop_type in['PTS', 'AST', 'PRA', 'P+A'] and pr_rank >= 25: return "🔥 Elite P&R"
            if is_big and prop_type in['PTS', 'REB', 'PRA', 'P+R'] and rim_rank >= 25: return "🔥 Elite Rim"
            if prop_type in['3PT', 'FG3M'] and spot_rank >= 25: return "🔥 Elite Spot-Up"
            if prop_type in['3PT', 'FG3M'] and perim_rank >= 25: return "🔥 Weak Perim Def"
            if is_handler and prop_type in['PTS', 'AST', 'PRA', 'P+A'] and pr_rank <= 5: return "⚠️ P&R Trap"
            if is_big and prop_type in['PTS', 'REB', 'PRA', 'P+R'] and rim_rank <= 5: return "⚠️ Rim Trap"
            if prop_type in['3PT', 'FG3M'] and spot_rank <= 5: return "⚠️ Spot-Up Trap"
            if prop_type in['3PT', 'FG3M'] and perim_rank <= 5: return "🔄 PIVOT TO 2PT"
        elif bet_type == 'Under':
            if is_handler and prop_type in['PTS', 'AST', 'PRA', 'P+A'] and pr_rank <= 5: return "🔒 Elite Under (P&R)"
            if is_big and prop_type in['PTS', 'REB', 'PRA', 'P+R'] and rim_rank <= 5: return "🔒 Elite Under (Rim)"
            if prop_type in['3PT', 'FG3M'] and spot_rank <= 5: return "🔒 Elite Under (Spot)"
            if prop_type in['3PT', 'FG3M'] and perim_rank <= 5: return "🔒 Elite Under (Perim)"
            if is_handler and prop_type in['PTS', 'AST', 'PRA', 'P+A'] and pr_rank >= 25: return "⚠️ Under Trap (P&R)"
            if is_big and prop_type in['PTS', 'REB', 'PRA', 'P+R'] and rim_rank >= 25: return "⚠️ Under Trap (Rim)"
        return "-"

    merged['Adv Matchup'] = merged.apply(get_adv_matchup_signal, axis=1)

    def divide_or_zero(n, d): return n / d if d != 0 else 0

    def calculate_base_score(row):
        score = 50; bet = row['Bet']; sl = row['Sharp Line']; line = row['Line']
        is_star = row.get('Is_Star', False); trend = row.get('TREND_PTS', 0)
        opp_rank = str(row.get('Opp Rank', ''))
        h2h_str = row.get('vs Opp', '0/0')

        # [NEW] H2H Crusher Bonus (Dort Rule)
        if '/' in h2h_str:
            try:
                hits, total = map(int, h2h_str.split('/'))
                if total >= 2 and hits == total: score += 25 # Perfect H2H record
            except: pass

        # [NEW] Targeted Defense Adjustment (The Brunson Rule)
        if trend > 5.0 and not is_star:
            score -= 25 # Role players get focused and falter
        elif trend > 5.0 and is_star:
            score += 10 # Stars thrive on the attention

        # [FIX] Star Resilience Logic: Stars overcome hard matchups
        if bet == 'Over' and "(Hard)" in opp_rank:
            if is_star: score -= 10 # Star discount
            else: score -= 40 # Role players falter

        if not pd.isna(sl):
            market_diff = abs(sl - line)
            if (bet == 'Over' and sl < line) or (bet == 'Under' and sl > line):
                score -= 40
            elif (bet == 'Over' and sl > line) or (bet == 'Under' and sl < line):
                score += 20
            if market_diff >= 1.5:
                score += 30
        return score

    merged['Score'] = merged.apply(calculate_base_score, axis=1)

    def classify_bet(row):
        sl = row.get('Sharp Line'); line = row.get('Line')
        is_star = row.get('Is_Star', False); trend = row.get('TREND_PTS', 0)
        alpha = row.get('Model Alpha', 0)
        h2h_str = row.get('vs Opp', '0/0')

        # [NEW] H2H Crusher Classification
        if '/' in h2h_str:
            try:
                hits, total = map(int, h2h_str.split('/'))
                if total >= 3 and hits == total: return "🔥 H2H CRUSHER"
            except: pass

        # [NEW] Atmosphere Boost (MSG/Knicks Factor)
        high_intensity_venues = ['New York Knicks', 'Boston Celtics', 'Philadelphia 76ers', 'Denver Nuggets']
        if row['Home_Team_Full'] in high_intensity_venues and row['Team'] == get_team_abbr_map().get(row['Home_Team_Full']):
            if row['Score'] >= 50: return "🏟️ VENUE BOOST"

        if is_star and trend > 2.0 and alpha > 1.5: return "🔥 STANDOUT STAR"
        if not pd.isna(sl) and not pd.isna(line) and abs(sl - line) >= 1.5: return "🚨 SHARP ALERT"

        if row['Score'] >= 70: return "💎 ALPHA (Lock)"
        if row['Score'] >= 50: return "📈 BETA (Parlay)"
        if row['Score'] >= 30: return "🤖 GAMMA (Spec)"
        return "PASS"

    merged['Bet Class'] = merged.apply(classify_bet, axis=1)

    # --- FD MY WAY CALCULATION (UPDATED WITH SHARP VALIDATION & TIGHTER LINES) ---
    def build_fd_my_way(df):
        my_way_rows =[]
        for _, row in df.iterrows():
            prop = row.get('Prop'); proj = row.get('Projection', 0); bet = row.get('Bet', '')
            sl = row.get('Sharp Line'); line = row.get('Line')
            h2h_str = row.get('vs Opp', '0/0')

            # [FIX] Sharp Validation: Filter out plays where Sharps disagree significantly
            if not pd.isna(sl):
                if bet == 'Over' and sl < (line - 0.5): continue
                if bet == 'Under' and sl > (line + 0.5): continue

            # [FIX] Tightened Alt-Lines (0.75 MAE instead of 1.2)
            mae = mae_values.get(SIMPLE_PROP_MAP.get(prop, 'PTS'), 1)
            alt = round(proj - (0.75 * mae), 1) if bet == 'Over' else round(proj + (0.75 * mae), 1)

            evidence = ["Statistically Safe"]
            if not pd.isna(sl) and ((bet == 'Over' and sl > line) or (bet == 'Under' and sl < line)): evidence.append("💎 Sharp Support")
            if row.get('Is_Star'): evidence.append("⭐ Star Power")
            if row['Bet Class'] == "🏟️ VENUE BOOST": evidence.append("🏟️ Home Intensity")
            if '/' in h2h_str:
                try:
                    hits, total = map(int, h2h_str.split('/'))
                    if total >= 2 and hits == total: evidence.append(f"🎯 H2H {h2h_str}")
                except: pass

            my_way_rows.append({
                'Game_Time': row.get('Game_Time'), 'Game': row.get('Game'), 'Player': row.get('Player'),
                'Prop': prop, 'Bet': bet, 'Alt Line': f"🟢 {alt}" if bet == 'Over' else f"🔴 {alt}",
                'Standard Line': line, 'Projection': proj,
                'Confidence': "85%", 'Supporting Evidence': ", ".join(evidence)
            })
        return pd.DataFrame(my_way_rows)

    if run_mode == 'final':
        ts_str = live_now_et.strftime('%-m-%-d-%y %-I:%M%p').lower()
        out_file = os.path.join(OUTPUT_DIR, f"{ts_str} NBA Pro Dashboard.xlsx")
    else:
        out_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Pro Dashboard.xlsx")

    # [FIX] Expanded Column Width values to prevent standard Excel render clipping
    col_widths = {
        'Game_Time': 14.5, 'Game': 12.0, 'Status': 10.0, 'Experience': 14.0, 'Team': 10.0, 'Player': 20.0,
        'Prop': 10.0, 'Line': 9.0, 'Bet': 8.5, 'Odds': 10.0, 'Sharp Line': 13.5,
        'Market Disc.': 15.0, 'Proj vs Sharp': 16.0, 'Projection': 13.5, 'Model Alpha': 15.0, 'Edge (%)': 12.0,
        'Consistency': 16.0, 'Opp Rank': 13.0, 'Adv Matchup': 20.0, 'Streak': 10.5, 'L10': 9.0, 'vs Opp': 11.0, 'Shot Quality': 16.0,
        'Game Move': 15.0, 'Game CLV': 15.0, 'Sharp Signal': 17.0, 'Ranking': 18.0, 'Score': 8.0, 'Bet Class': 19.0
    }

    with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
        workbook = writer.book
        header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1})
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        orange_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        blue_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        purple_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        perfect_fmt = workbook.add_format({'bg_color': '#00FF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})
        yellow_fmt = workbook.add_format({'bg_color': '#FFFF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1})

        green_row_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        blue_row_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        red_row_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        yellow_row_fmt = workbook.add_format({'bg_color': '#FFFF00', 'font_color': '#000000', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        purple_row_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'left', 'valign': 'vcenter', 'border': 1})

        ws_howto = workbook.add_worksheet("HOW TO USE")
        ws_howto.set_column(0, 0, 35.8); ws_howto.set_column(1, 1, 101.0); ws_howto.freeze_panes(1, 0)
        row_idx = 0
        ws_howto.write(row_idx, 0, "Item", header_fmt); ws_howto.write(row_idx, 1, "Description", header_fmt); row_idx += 1

        ws_howto.write(row_idx, 0, "💎 ALPHA (Lock)", perfect_fmt); ws_howto.write(row_idx, 1, "Model, Sharps, and Price all align. 2-4 plays per day max.", perfect_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🔥 STANDOUT STAR", perfect_fmt); ws_howto.write(row_idx, 1, "Star player heating up with high model alpha.", perfect_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🔥 H2H CRUSHER", perfect_fmt); ws_howto.write(row_idx, 1, "Player has a perfect historical record against this opponent.", perfect_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🏟️ VENUE BOOST", blue_fmt); ws_howto.write(row_idx, 1, "Home court intensity boost (MSG, TD Garden, etc).", blue_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "📈 BETA (Parlay Leg)", green_fmt); ws_howto.write(row_idx, 1, "Strong sharp movement confirmed by professional markets.", green_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🤖 GAMMA (Speculative)", yellow_fmt); ws_howto.write(row_idx, 1, "High model edge before the sharps have moved the line.", yellow_fmt); row_idx += 1
        row_idx += 1

        ws_howto.write(row_idx, 0, "PARLAY SCORE HIGHLIGHTS (Player Name)", header_fmt); ws_howto.write(row_idx, 1, "Master Score combining Win Prob + Edge + Matchup + Form + Sharps.", header_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Bold Green (Score > 75)", perfect_fmt); ws_howto.write(row_idx, 1, "The Parlay Anchors. High probability, strong edge, elite matchup.", perfect_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Green (Score 55-75)", green_row_fmt); ws_howto.write(row_idx, 1, "Strong Legs. Great secondary pieces for standard parlays.", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Yellow (Score 30-55)", yellow_row_fmt); ws_howto.write(row_idx, 1, "Lotto Legs / Caution. High ceiling but comes with a matchup or volatility risk.", yellow_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Deleted / Hidden Bets", red_row_fmt); ws_howto.write(row_idx, 1, "NOTE: Any bet with a Score < 30, a Trap Matchup, or a Lottery Edge is now auto-deleted to protect you.", red_row_fmt); row_idx += 1
        row_idx += 1

        ws_howto.write(row_idx, 0, "NEW: Advanced Matchup (Synergy & Tracking)", header_fmt); ws_howto.write(row_idx, 1, "Uses Synergy Play-Type & NBA Tracking Cameras to expose defensive weaknesses.", header_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🔥 Elite P&R / Elite Rim / Weak Perim", green_row_fmt); ws_howto.write(row_idx, 1, "Player's play-style directly attacks the opponent's biggest weakness. (+20 Score)", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🔒 Elite Under", green_row_fmt); ws_howto.write(row_idx, 1, "Model projects Under, and the opponent is Top 5 at defending this exact play type. (+20 Score)", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "⚠️ Trap Matchup", red_row_fmt); ws_howto.write(row_idx, 1, "Model projects Over, but the opponent is Top 5 at defending this exact play type. (-25 Score)", red_row_fmt); row_idx += 1
        row_idx += 1

        ws_howto.write(row_idx, 0, "ACTIVE SHOT QUALITY", header_fmt); ws_howto.write(row_idx, 1, "Now mathematically impacts the Master Score to solidify locks and expose fake streaks.", header_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🎯 High SQ", green_row_fmt); ws_howto.write(row_idx, 1, "Player gets a lot of Wide Open shots. Boosts Over confidence. (+15 Score)", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "🔥 Due", green_row_fmt); ws_howto.write(row_idx, 1, "Player is getting open looks but shooting poorly. Positive regression is coming. (+25 Score)", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "⚠️ Luck Risk", yellow_row_fmt); ws_howto.write(row_idx, 1, "Player is shooting unsustainably hot on highly contested shots. Negative regression imminent.", yellow_row_fmt); row_idx += 1
        row_idx += 1

        ws_howto.write(row_idx, 0, "Movement: 🔥 Steam Chase", yellow_row_fmt); ws_howto.write(row_idx, 1, "Massive, sudden line move (>1.5 pts) caused by sharp money. Strongest confirmation signal.", yellow_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "CLV (Closing Line Value)", green_row_fmt); ws_howto.write(row_idx, 1, "Green (>0) means you beat the market. Red (<0) means the line moved against you.", green_row_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Signal: SHARP MISMATCH", purple_row_fmt); ws_howto.write(row_idx, 1, "Retail line significantly differs from Sharp Book (Pinnacle/Betfair) line.", purple_row_fmt); row_idx += 1
        row_idx += 1

        ws_howto.write(row_idx, 0, "GAME LINE PICKS", header_fmt); ws_howto.write(row_idx, 1, "How to read the Game_Lines sheet.", header_fmt); row_idx += 1
        ws_howto.write(row_idx, 0, "Spread / Total Pick", green_row_fmt); ws_howto.write(row_idx, 1, "If it displays a team (e.g. Houston -2.5), the model edge is > 1.0 point. If it shows '-', the edge is too small to bet.", green_row_fmt); row_idx += 1

        sheet_config =['DraftKings_EV_Bets', 'DraftKings_EV_SGP', 'FanDuel_EV_Bets', 'FanDuel_EV_SGP']
        for sheet_name in sheet_config:
            bookmaker = 'DraftKings' if 'DraftKings' in sheet_name else 'FanDuel'
            is_sgp = 'SGP' in sheet_name

            # If in final mode, keep strict filters. If in initial capture/other modes, keep it open to ensure comparison baseline is saved.
            if run_mode == 'final':
                book_df = merged[(merged['Bookmaker'] == bookmaker) & (merged['Bet'] != 'Pass') & (merged['Score'] >= 30) & (merged['Bet Class'] != 'PASS')].copy()
            else:
                book_df = merged[(merged['Bookmaker'] == bookmaker) & (merged['Bet'] != 'Pass')].copy()

            if not book_df.empty:
                out_cols =[
                    'Game_Time', 'Game', 'Status', 'Experience', 'Team', 'Player', 'Prop',
                    'Line', 'Bet', 'Odds', 'Sharp Line', 'Market Disc.', 'Proj vs Sharp',
                    'Projection', 'Model Alpha', 'Edge (%)',
                    'Consistency', 'Opp Rank', 'Adv Matchup', 'Streak', 'L10', 'vs Opp',
                    'Shot Quality',
                    'Game Move', 'Game CLV',
                    'Sharp Signal', 'Ranking', 'Score', 'Bet Class'
                ]

                if is_sgp: final_df = book_df.sort_values(['Sort_Time', 'Game', 'Score'], ascending=[True, True, False]).reindex(columns=out_cols)
                else: final_df = book_df.sort_values(['Score'], ascending=[False]).reindex(columns=out_cols)

                final_df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1, header=False)
                ws = writer.sheets[sheet_name]
                ws.freeze_panes(1, 6)
                ws.autofilter(0, 0, len(final_df), len(out_cols) - 1)

                for col_idx, val in enumerate(final_df.columns):
                    ws.write(0, col_idx, val, header_fmt)
                    ws.set_column(col_idx, col_idx, col_widths.get(val, 10), center_fmt)

                last_row = len(final_df) + 1
                def get_col(name): return xl_col_to_name(out_cols.index(name))

                ws.set_column(out_cols.index('Score'), out_cols.index('Score'), None, None, {'hidden': True})

                line_col = get_col('Line'); bet_col = get_col('Bet'); sharp_col = get_col('Sharp Line')
                proj_col = get_col('Projection'); opp_rank_col = get_col('Opp Rank'); player_col = get_col('Player')
                score_col = get_col('Score'); odds_col = get_col('Odds'); adv_col = get_col('Adv Matchup')
                class_col = get_col('Bet Class'); v_opp_col = get_col('vs Opp')

                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'STAR', 'format': perfect_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'BOOST', 'format': blue_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'ALPHA', 'format': perfect_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'BETA', 'format': green_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'GAMMA', 'format': yellow_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'ALERT', 'format': orange_fmt})
                ws.conditional_format(f'{class_col}2:{class_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'CRUSHER', 'format': perfect_fmt})

                ws.conditional_format(f'{player_col}2:{player_col}{last_row}', {'type': 'formula', 'criteria': f'=${score_col}2>=75', 'format': perfect_fmt})
                ws.conditional_format(f'{player_col}2:{player_col}{last_row}', {'type': 'formula', 'criteria': f'=AND(${score_col}2>=55, ${score_col}2<75)', 'format': green_fmt})
                ws.conditional_format(f'{player_col}2:{player_col}{last_row}', {'type': 'formula', 'criteria': f'=AND(${score_col}2>=30, ${score_col}2<55)', 'format': yellow_fmt})

                ws.conditional_format(f'{opp_rank_col}2:{opp_rank_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Hard', 'format': green_fmt})
                ws.conditional_format(f'{opp_rank_col}2:{opp_rank_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Easy', 'format': green_fmt})

                green_matchup_criteria = f'=OR(AND(${bet_col}2="Over", ISNUMBER(SEARCH("Easy", ${opp_rank_col}2))), AND(${bet_col}2="Under", ISNUMBER(SEARCH("Hard", ${opp_rank_col}2))))'
                ws.conditional_format(f'{bet_col}2:{bet_col}{last_row}', {'type': 'formula', 'criteria': green_matchup_criteria, 'format': green_fmt})

                ws.conditional_format(f'{adv_col}2:{adv_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': '🔥', 'format': green_fmt})
                ws.conditional_format(f'{adv_col}2:{adv_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': '🔒', 'format': green_fmt})
                ws.conditional_format(f'{adv_col}2:{adv_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': '🔄', 'format': yellow_fmt})

                ws.conditional_format(f'{odds_col}2:{odds_col}{last_row}', {'type': 'formula', 'criteria': f'=OR(LEFT(${odds_col}2,1)="+", AND(LEFT(${odds_col}2,1)="-", VALUE(SUBSTITUTE(${odds_col}2,"-",""))<=110))', 'format': green_fmt})
                ws.conditional_format(f'{odds_col}2:{odds_col}{last_row}', {'type': 'formula', 'criteria': f'=AND(LEFT(${odds_col}2,1)="-", VALUE(SUBSTITUTE(${odds_col}2,"-",""))>110, VALUE(SUBSTITUTE(${odds_col}2,"-",""))<=125)', 'format': yellow_fmt})
                ws.conditional_format(f'{odds_col}2:{odds_col}{last_row}', {'type': 'formula', 'criteria': f'=AND(LEFT(${odds_col}2,1)="-", VALUE(SUBSTITUTE(${odds_col}2,"-",""))>=135)', 'format': blue_fmt})

                ws.conditional_format(f'{sharp_col}2:{sharp_col}{last_row}', {'type': 'formula', 'criteria': f'=OR(AND(${bet_col}2="Over", ISNUMBER(${sharp_col}2), ${sharp_col}2>${line_col}2), AND(${bet_col}2="Under", ISNUMBER(${sharp_col}2), ${sharp_col}2<${line_col}2))', 'format': purple_fmt})

                proj_vs_sharp_col = get_col('Proj vs Sharp')
                ws.conditional_format(f'{proj_vs_sharp_col}2:{proj_vs_sharp_col}{last_row}', {
                    'type': 'formula',
                    'criteria': f'=OR(AND(${bet_col}2="Over", ${proj_vs_sharp_col}2>=1.5), AND(${bet_col}2="Under", ${proj_vs_sharp_col}2<=-1.5))',
                    'format': green_fmt
                })

                sq_col = get_col('Shot Quality')
                ws.conditional_format(f'{sq_col}2:{sq_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'High SQ', 'format': green_fmt})
                ws.conditional_format(f'{sq_col}2:{sq_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Due', 'format': green_fmt})
                ws.conditional_format(f'{sq_col}2:{sq_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Luck Risk', 'format': yellow_fmt})

                streak_col = get_col('Streak')
                ws.conditional_format(f'{streak_col}2:{streak_col}{last_row}', {
                    'type': 'formula',
                    'criteria': f'=VALUE(SUBSTITUTE(${streak_col}2, "W", ""))>=3',
                    'format': green_fmt
                })

                ws.conditional_format(f'{get_col("Sharp Signal")}2:{get_col("Sharp Signal")}{last_row}', {'type': 'text', 'criteria': 'not containing', 'value': '-', 'format': purple_fmt})
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'cell', 'criteria': '>', 'value': 10, 'format': green_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Profitable', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'PRIME', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Validated', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Safest', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Solid', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Strong Value', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'High', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Status")}2:{get_col("Status")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Questionable', 'format': orange_fmt})
                ws.conditional_format(f'{v_opp_col}2:{v_opp_col}{last_row}', {'type': 'formula', 'criteria': f'=AND(IFERROR(FIND("/", ${v_opp_col}2)>0, FALSE), IFERROR(VALUE(LEFT(${v_opp_col}2, FIND("/", ${v_opp_col}2)-1)) = VALUE(MID(${v_opp_col}2, FIND("/", ${v_opp_col}2)+1, 10)), FALSE), IFERROR(VALUE(MID(${v_opp_col}2, FIND("/", ${v_opp_col}2)+1, 10)) > 0, FALSE))', 'format': perfect_fmt})

                for trend_val in['8/10', '9/10', '10/10']:
                    ws.conditional_format(f'{get_col("L10")}2:{get_col("L10")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': trend_val, 'format': green_fmt})
                ws.conditional_format(f'{get_col("Game Move")}2:{get_col("Game Move")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Steam', 'format': yellow_fmt})
                ws.conditional_format(f'{get_col("Game Move")}2:{get_col("Game Move")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'RLM', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Game CLV")}2:{get_col("Game CLV")}{last_row}', {'type': 'cell', 'criteria': '>', 'value': 0, 'format': green_fmt})

            else: writer.book.add_worksheet(sheet_name).write(0, 0, "No bets found.")

        # --- FD MY WAY CALCULATION (UPDATED WITH SHARP VALIDATION & TIGHTER LINES) ---
        fd_df = merged[(merged['Bookmaker'] == 'FanDuel') & (merged['Bet'] != 'Pass')].copy()
        if not fd_df.empty:
            my_way_df = build_fd_my_way(fd_df)
            if not my_way_df.empty:
                my_way_df = my_way_df.sort_values(by='Confidence', ascending=False)
                my_way_df.to_excel(writer, sheet_name='FD My Way', index=False, startrow=1, header=False)
                ws_mw = writer.sheets['FD My Way']
                ws_mw.freeze_panes(1, 3)
                ws_mw.autofilter(0, 0, len(my_way_df), len(my_way_df.columns) - 1)
                my_way_widths = {
                    'Game_Time': 14.5, 'Game': 12.0, 'Player': 20.0,
                    'Prop': 10.0, 'Bet': 8.5, 'Alt Line': 11.5, 'Standard Line': 15.5,
                    'Projection': 13.5, 'Confidence': 14.0, 'Supporting Evidence': 36
                }
                for col_idx, val in enumerate(my_way_df.columns):
                    ws_mw.write(0, col_idx, val, header_fmt)
                    ws_mw.set_column(col_idx, col_idx, my_way_widths.get(val, 16), center_fmt)

                lr_mw = len(my_way_df) + 1
                c_conf = xl_col_to_name(my_way_df.columns.get_loc('Confidence'))
                c_alt = xl_col_to_name(my_way_df.columns.get_loc('Alt Line'))

                ws_mw.conditional_format(f'{c_conf}2:{c_conf}{lr_mw}', {'type': 'cell', 'criteria': '>=', 'value': '"85%"', 'format': perfect_fmt})
                ws_mw.conditional_format(f'{c_alt}2:{c_alt}{lr_mw}', {'type': 'text', 'criteria': 'containing', 'value': '🟢', 'format': green_fmt})
                ws_mw.conditional_format(f'{c_alt}2:{c_alt}{lr_mw}', {'type': 'text', 'criteria': 'containing', 'value': '🔴', 'format': red_fmt})

        if game_lines_rows:
            gl_df = pd.DataFrame(game_lines_rows)
            gl_cols =['Time', 'Game', 'Retail Spread', 'Sharp Spread', 'Model Spread', 'Retail Total', 'Sharp Total', 'Model Total', 'Retail ML', 'Model Win %', 'Home_Team_Full', 'Game Move', 'Game CLV']
            gl_df = gl_df.reindex(columns=gl_cols)
            gl_df.to_excel(writer, sheet_name='Game_Lines', index=False); ws_gl = writer.sheets['Game_Lines']
            ws_gl.freeze_panes(1, 0); ws_gl.autofilter(0, 0, len(gl_df), len(gl_cols) - 1)
            for col_idx, val in enumerate(gl_df.columns):
                ws_gl.write(0, col_idx, val, header_fmt)
                ws_gl.set_column(col_idx, col_idx, 14, center_fmt)

    print(f"✅ Exported: {out_file}")

if __name__ == "__main__":
    run_daily_projections()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 17.5 MB/s eta 0:00:00

🧠 PHASE 0: PRE-FLIGHT ACCURACY AUDIT (Last 1,000 Games)
Prop Type  Avg Error (MAE)  Volatility (RMSE) Error %
      3PT             0.84               1.20   73.8%
      AST             1.21               1.61   53.5%
      REB             1.80               2.35   45.0%
      R+A             2.39               3.06   38.2%
      PTS             4.32               5.68   42.2%
      P+A             4.66               6.08   37.3%
      P+R             5.18               6.70   36.4%
      PRA             5.56               7.13   33.7%


📅 Running Live Mode. Current ET Time: 2026-06-08 10:17:57
Enter run mode ('i' for initial capture, 'f' for final report, 't' for training): i

🔵 INITIAL RUN DETECTED. Will save opening lines to CSV.
Running projections for 2026-06-08 (Season: 2025-26)
✅ All models, imputers, and MAE values loa

Fetching Team Rosters for Injuries: 100%|██████████| 30/30 [00:02<00:00, 10.28it/s]


   ✅ Successfully fetched 126 injury entries.
   ℹ Detected 126 ruled-out players. Adjusting team usage automatically...

--- Preparing Context & Stats ---
✅ Successfully fetched season stats.
   🛡️ Fetching Opponent Defense Rankings (Raw & Calculated)...
   🛡️ Fetching Advanced Team Defense (Synergy & Rim) for the entire league...
   🧮 Calculating Feng/Massey Ratings...
   📊 Fetching Team Metrics for 2025-26...
--- Fetching Schedule for 2026-06-08 via API (ScoreboardV3) ---
✅ Found 1 valid games on the schedule.
   🎯 Fetching League-Wide Shot Quality Data (General & Wide Open) for 2025-26...

--- Running Initial Projections & Monte Carlo Sims ---


Generating Projections: 100%|██████████| 2/2 [00:44<00:00, 22.06s/it]


✅ Generated 32 player projections.

--- Fetching Market Odds Data ---
ℹ No earlier run found for TODAY (06-08-2026). This is likely the 'Pro Window' (First Run).
✅ Exported: /content/drive/MyDrive/NBA/Predictions/06-08-2026 NBA Pro Dashboard.xlsx
